In [ ]:
# ============================================================
# CELL 1 — CONFIGURATION + UPLOAD / LOAD EXCEL
# ============================================================

# -----------------------------
# 1) USER SETTINGS
# -----------------------------
NOTEBOOK_MODE = "COLAB"   # "COLAB" or "LOCAL"

LOCAL_EXCEL_PATH = "Tres Cantos for optimization Residential.xlsx"

# Required sheet names
EXPECTED_SHEETS = [
    "parcels",
    "complementarity_weights",
    "visit_frequency_weights",
    "distance_weights"
]

# Main inter-parcel search radius (meters)
MAX_DISTANCE = 1200.0

# Random seed for reproducibility
RANDOM_SEED = 42


# -----------------------------
# 2) IMPORTS NEEDED FOR LOADING
# -----------------------------
import os
import random
import numpy as np
import pandas as pd

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# -----------------------------
# 3) LOAD FILE
# -----------------------------
EXCEL_PATH = None

if NOTEBOOK_MODE == "COLAB":
    try:
        from google.colab import files
    except Exception:
        raise RuntimeError(
            "NOTEBOOK_MODE is set to 'COLAB' but google.colab is not available. "
            "If you are running locally, change NOTEBOOK_MODE to 'LOCAL'."
        )

    uploaded = files.upload()

    if len(uploaded) == 0:
        raise ValueError("No file uploaded. Please upload one Excel workbook.")

    EXCEL_PATH = list(uploaded.keys())[0]

elif NOTEBOOK_MODE == "LOCAL":
    EXCEL_PATH = LOCAL_EXCEL_PATH

    if not os.path.exists(EXCEL_PATH):
        raise FileNotFoundError(
            f"Excel file not found:\n{EXCEL_PATH}\n\n"
            "Check the file name/path or switch NOTEBOOK_MODE to 'COLAB'."
        )

else:
    raise ValueError("NOTEBOOK_MODE must be either 'COLAB' or 'LOCAL'.")


# -----------------------------
# 4) QUICK FILE CHECK
# -----------------------------
if not EXCEL_PATH.lower().endswith((".xlsx", ".xlsm", ".xls")):
    raise ValueError(
        f"The selected file does not look like an Excel workbook: {EXCEL_PATH}"
    )

xls = pd.ExcelFile(EXCEL_PATH)
found_sheets = xls.sheet_names

missing_sheets = [s for s in EXPECTED_SHEETS if s not in found_sheets]

if missing_sheets:
    raise ValueError(
        "Missing required sheet(s): "
        + ", ".join(missing_sheets)
        + "\n\nExpected sheets:\n- "
        + "\n- ".join(EXPECTED_SHEETS)
        + "\n\nFound sheets:\n- "
        + "\n- ".join(found_sheets)
    )


# -----------------------------
# 5) LOAD REQUIRED SHEETS
# -----------------------------
parcels_df = pd.read_excel(EXCEL_PATH, sheet_name="parcels")
complementarity_weights_df = pd.read_excel(EXCEL_PATH, sheet_name="complementarity_weights")
visit_frequency_weights_df = pd.read_excel(EXCEL_PATH, sheet_name="visit_frequency_weights")
distance_weights_df = pd.read_excel(EXCEL_PATH, sheet_name="distance_weights")


# -----------------------------
# 6) QUICK SUMMARY TO STUDENTS
# -----------------------------
print("✅ Excel file loaded successfully")
print(f"File: {EXCEL_PATH}")
print("")
print("Sheets found:")
for s in found_sheets:
    print(f" - {s}")

print("")
print("Quick summary:")
print(f" - Parcels rows: {len(parcels_df)}")
print(f" - Complementarity rows: {len(complementarity_weights_df)}")
print(f" - Visit frequency rows: {len(visit_frequency_weights_df)}")
print(f" - Distance weight rows: {len(distance_weights_df)}")
print(f" - MAX_DISTANCE: {MAX_DISTANCE} m")
print("")

print("Preview of parcels sheet:")
display(parcels_df.head(5))

In [ ]:
# ============================================================
# CELL 2 — AUTOMATIC PREPARATION OF MODEL INPUTS
# ============================================================

# -----------------------------
# 0) IMPORTS
# -----------------------------
import os
import math
import random
import time
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# -----------------------------
# 1) BASIC VALIDATION HELPERS
# -----------------------------
def require_columns(df: pd.DataFrame, required: List[str], sheet_name: str):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"❌ Sheet '{sheet_name}' is missing required columns: {missing}")

def coerce_numeric(df: pd.DataFrame, cols: List[str], sheet_name: str):
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        if df[c].isna().any():
            bad_n = int(df[c].isna().sum())
            raise ValueError(
                f"❌ Sheet '{sheet_name}' column '{c}' contains {bad_n} non-numeric or missing values."
            )

def normalize_text_cols(df: pd.DataFrame, cols: List[str]):
    for c in cols:
        df[c] = df[c].astype(str).str.strip()


# -----------------------------
# 2) VALIDATE PARCELS SHEET
# -----------------------------
PARCEL_REQUIRED = [
    "parcel_id",
    "longitude",
    "latitude",
    "parcel_total_functions_area"
]
require_columns(parcels_df, PARCEL_REQUIRED, "parcels")

# Stable structure rule from your original notebook:
# first 6 columns are base columns, functions start at index 6
if parcels_df.shape[1] <= 6:
    raise ValueError(
        "❌ 'parcels' sheet must have more than 6 columns. "
        "Function columns are expected to start at column index 6."
    )

BASE_COLS = list(parcels_df.columns[:6])
FUNCTION_COLS = list(parcels_df.columns[6:])

if len(FUNCTION_COLS) == 0:
    raise ValueError("❌ No function columns were detected in 'parcels' sheet.")

# Clean parcel identifiers
parcels_df["parcel_id"] = parcels_df["parcel_id"].astype(str).str.strip()

if parcels_df["parcel_id"].isna().any():
    raise ValueError("❌ 'parcels.parcel_id' contains missing values.")

if parcels_df["parcel_id"].duplicated().any():
    dups = parcels_df.loc[parcels_df["parcel_id"].duplicated(), "parcel_id"].head(10).tolist()
    raise ValueError(f"❌ Duplicate parcel_id values found. Examples: {dups}")

# Numeric checks
parcel_numeric_cols = ["longitude", "latitude", "parcel_total_functions_area"] + FUNCTION_COLS
coerce_numeric(parcels_df, parcel_numeric_cols, "parcels")

# Optional but important for later FAR logic
HAS_PARCEL_GROUND_AREA = "parcel_ground_area" in parcels_df.columns
if HAS_PARCEL_GROUND_AREA:
    coerce_numeric(parcels_df, ["parcel_ground_area"], "parcels")

# Negative values check for function areas
neg_mask = (parcels_df[FUNCTION_COLS] < 0).any(axis=1)
if neg_mask.any():
    bad_ids = parcels_df.loc[neg_mask, "parcel_id"].head(10).tolist()
    raise ValueError(
        f"❌ Negative function areas found in 'parcels' sheet. Example parcel_id values: {bad_ids}"
    )


# -----------------------------
# 3) VALIDATE WEIGHT TABLES
# -----------------------------
# ---- complementarity_weights ----
CW_REQUIRED = ["function_type_1", "function_type_2", "complementarity_weight"]
require_columns(complementarity_weights_df, CW_REQUIRED, "complementarity_weights")
normalize_text_cols(complementarity_weights_df, ["function_type_1", "function_type_2"])
coerce_numeric(complementarity_weights_df, ["complementarity_weight"], "complementarity_weights")

if complementarity_weights_df[CW_REQUIRED].isna().any().any():
    raise ValueError("❌ 'complementarity_weights' has missing values in required columns.")

# ---- visit_frequency_weights ----
VF_REQUIRED = ["function_type", "visit_frequency_weight"]
VF_VARIANTS = [
    ["function_type", "visit_frequency_weight"],   # preferred
    ["function_type", "visit_frequency_weights"],  # plural
    ["function_type", "visit_frequency"],          # short
    ["function_type", "usage_intensity_weight"],   # workbook variant
    ["function_type", "weight"],                   # generic
]

matched = None
for cols in VF_VARIANTS:
    if all(c in visit_frequency_weights_df.columns for c in cols):
        matched = cols
        break

if matched is None:
    raise ValueError(
        "❌ Sheet 'visit_frequency_weights' must contain columns like:\n"
        "  - ['function_type', 'visit_frequency_weight'] (preferred)\n"
        "  - or one of these variants: "
        f"{VF_VARIANTS}\n"
        f"Found columns: {list(visit_frequency_weights_df.columns)}"
    )

if matched[1] != "visit_frequency_weight":
    visit_frequency_weights_df = visit_frequency_weights_df.rename(
        columns={matched[1]: "visit_frequency_weight"}
    )
    print(f"ℹ️ Renamed '{matched[1]}' -> 'visit_frequency_weight' for internal consistency.")

require_columns(visit_frequency_weights_df, VF_REQUIRED, "visit_frequency_weights")
normalize_text_cols(visit_frequency_weights_df, ["function_type"])
coerce_numeric(visit_frequency_weights_df, ["visit_frequency_weight"], "visit_frequency_weights")

if visit_frequency_weights_df[VF_REQUIRED].isna().any().any():
    raise ValueError("❌ 'visit_frequency_weights' has missing values in required columns.")

# ---- distance_weights ----
# Original notebook uses: min_distance, max_distance, weight
DW_REQUIRED = ["min_distance", "max_distance", "weight"]
require_columns(distance_weights_df, DW_REQUIRED, "distance_weights")
coerce_numeric(distance_weights_df, ["min_distance", "max_distance", "weight"], "distance_weights")

if distance_weights_df[DW_REQUIRED].isna().any().any():
    raise ValueError("❌ 'distance_weights' has missing values in required columns.")

if not (distance_weights_df["min_distance"] < distance_weights_df["max_distance"]).all():
    raise ValueError("❌ 'distance_weights' has rows where min_distance >= max_distance.")

if distance_weights_df["max_distance"].max() < MAX_DISTANCE:
    raise ValueError(
        f"❌ 'distance_weights' does not cover MAX_DISTANCE={MAX_DISTANCE}. "
        f"Max max_distance in sheet is {distance_weights_df['max_distance'].max()}."
    )

distance_weights_df = distance_weights_df.sort_values(["min_distance", "max_distance"]).reset_index(drop=True)


# -----------------------------
# 4) CHECK FUNCTION NAME CONSISTENCY
# -----------------------------
parcel_function_set = set(FUNCTION_COLS)
cw_f1 = set(complementarity_weights_df["function_type_1"].unique())
cw_f2 = set(complementarity_weights_df["function_type_2"].unique())
vf_set = set(visit_frequency_weights_df["function_type"].unique())

missing_in_cw_left = sorted(parcel_function_set - cw_f1)
missing_in_cw_right = sorted(parcel_function_set - cw_f2)
missing_in_vf = sorted(parcel_function_set - vf_set)

extra_cw_left = sorted(cw_f1 - parcel_function_set)
extra_cw_right = sorted(cw_f2 - parcel_function_set)
extra_vf = sorted(vf_set - parcel_function_set)

print("Function consistency check")
print(f" - Functions in parcels sheet: {len(parcel_function_set)}")

if missing_in_cw_left:
    print(" - Warning: missing as function_type_1 in complementarity_weights:", missing_in_cw_left)
if missing_in_cw_right:
    print(" - Warning: missing as function_type_2 in complementarity_weights:", missing_in_cw_right)
if missing_in_vf:
    print(" - Warning: missing in visit_frequency_weights:", missing_in_vf)


# -----------------------------
# 5) BUILD CORE ARRAYS / LOOKUPS
# -----------------------------
parcel_ids = parcels_df["parcel_id"].tolist()
parcel_index = {pid: i for i, pid in enumerate(parcel_ids)}

coordinates = parcels_df[["longitude", "latitude"]].to_numpy(dtype=float)

function_cols = FUNCTION_COLS[:]
functions_matrix = parcels_df[function_cols].to_numpy(dtype=float)

parcel_total_functions_area = parcels_df["parcel_total_functions_area"].to_numpy(dtype=float)

if HAS_PARCEL_GROUND_AREA:
    parcel_ground_area = parcels_df["parcel_ground_area"].to_numpy(dtype=float)
else:
    parcel_ground_area = None

N = len(parcel_ids)
M = len(function_cols)

BASE_MATRIX = functions_matrix.copy()

vacant_mask = np.isclose(functions_matrix, 0.0).all(axis=1)

existing_function_count = (functions_matrix > 0).sum(axis=1)
dominant_function_idx = np.where(
    functions_matrix.sum(axis=1) > 0,
    np.argmax(functions_matrix, axis=1),
    -1
)
dominant_function = np.array(
    [function_cols[j] if j >= 0 else None for j in dominant_function_idx],
    dtype=object
)


# -----------------------------
# 6) BUILD WEIGHT STRUCTURES
# -----------------------------
visit_frequency_weights: Dict[str, float] = (
    visit_frequency_weights_df
    .set_index("function_type")["visit_frequency_weight"]
    .to_dict()
)

complementarity_weights: Dict[Tuple[str, str], float] = (
    complementarity_weights_df
    .set_index(["function_type_1", "function_type_2"])["complementarity_weight"]
    .to_dict()
)

# Original notebook uses "weight" here
distance_weights = distance_weights_df[["min_distance", "max_distance", "weight"]].to_numpy(dtype=float)

func_idx = {f: j for j, f in enumerate(function_cols)}

visit_freq_vec = np.array(
    [float(visit_frequency_weights.get(f, 1.0)) for f in function_cols],
    dtype=float
)

comp_weight_matrix = np.zeros((M, M), dtype=float)
for f1 in function_cols:
    for f2 in function_cols:
        j1 = func_idx[f1]
        j2 = func_idx[f2]
        comp_weight_matrix[j1, j2] = float(complementarity_weights.get((f1, f2), 0.0))


# -----------------------------
# 7) KD-TREE + SPATIAL NEIGHBORS
# -----------------------------
kd_tree = cKDTree(coordinates)

def get_distance_weight(dist: float, distance_weights_arr: np.ndarray) -> float:
    """
    distance_weights_arr rows: [min_distance, max_distance, weight]
    Returns 0.0 if out of range.
    """
    for mn, mx, w in distance_weights_arr:
        if mn <= dist < mx:
            return float(w)
    return 0.0

neighbors_idx: List[np.ndarray] = []
neighbors_dist: List[np.ndarray] = []
neighbors_dist_w: List[np.ndarray] = []

t0 = time.time()

for i in range(N):
    idxs = kd_tree.query_ball_point(coordinates[i], r=MAX_DISTANCE)

    if i in idxs:
        idxs.remove(i)

    idxs_arr = np.array(idxs, dtype=int)

    if idxs_arr.size == 0:
        neighbors_idx.append(np.array([], dtype=int))
        neighbors_dist.append(np.array([], dtype=float))
        neighbors_dist_w.append(np.array([], dtype=float))
        continue

    dists = np.linalg.norm(coordinates[idxs_arr] - coordinates[i], axis=1)
    dw = np.array([get_distance_weight(d, distance_weights) for d in dists], dtype=float)

    neighbors_idx.append(idxs_arr)
    neighbors_dist.append(dists)
    neighbors_dist_w.append(dw)

elapsed = time.time() - t0


# -----------------------------
# 8) BASELINE DIAGNOSTIC HELPERS
# -----------------------------
def get_base_alloc(parcel_id: str) -> Dict[str, float]:
    i = parcel_index[parcel_id]
    return {f: float(BASE_MATRIX[i, j]) for j, f in enumerate(function_cols)}

def get_actual_alloc_from_row(row: np.ndarray) -> Dict[str, float]:
    return {f: float(row[j]) for j, f in enumerate(function_cols)}

def row_total_area(row: np.ndarray) -> float:
    return float(np.sum(row))

def compute_far_from_row(row: np.ndarray, ground_area: float) -> float:
    if ground_area is None or ground_area <= 0:
        return np.nan
    return float(np.sum(row) / ground_area)

def baseline_interparcel_score_for_parcel(i: int) -> float:
    row_i = BASE_MATRIX[i]
    total_i = row_i.sum()

    if total_i <= 0:
        return 0.0

    score = 0.0
    nz_i = np.where(row_i > 0)[0]
    if len(nz_i) == 0:
        return 0.0

    for k, nb in enumerate(neighbors_idx[i]):
        row_nb = BASE_MATRIX[nb]
        nz_nb = np.where(row_nb > 0)[0]
        if len(nz_nb) == 0:
            continue

        dist_w = neighbors_dist_w[i][k]

        for a in nz_i:
            ai = row_i[a] / total_i
            vf = visit_freq_vec[a]
            for b in nz_nb:
                score += ai * float(row_nb[b]) * comp_weight_matrix[a, b] * vf * dist_w

    return float(score)

baseline_interparcel_scores = np.array(
    [baseline_interparcel_score_for_parcel(i) for i in range(N)],
    dtype=float
)


# -----------------------------
# 9) DATA BUNDLE FOR LATER CELLS
# -----------------------------
data_bundle = {
    "parcel_ids": parcel_ids,
    "parcel_index": parcel_index,
    "coordinates": coordinates,
    "function_cols": function_cols,
    "functions_matrix": functions_matrix,
    "BASE_MATRIX": BASE_MATRIX,
    "parcel_total_functions_area": parcel_total_functions_area,
    "parcel_ground_area": parcel_ground_area,
    "visit_frequency_weights": visit_frequency_weights,
    "complementarity_weights": complementarity_weights,
    "distance_weights": distance_weights,
    "func_idx": func_idx,
    "visit_freq_vec": visit_freq_vec,
    "comp_weight_matrix": comp_weight_matrix,
    "neighbors_idx": neighbors_idx,
    "neighbors_dist": neighbors_dist,
    "neighbors_dist_w": neighbors_dist_w,
    "vacant_mask": vacant_mask,
    "existing_function_count": existing_function_count,
    "dominant_function": dominant_function,
    "baseline_interparcel_scores": baseline_interparcel_scores,
}


# -----------------------------
# 10) SUMMARY
# -----------------------------
print("✅ Cell 2 completed successfully")
print("")
print("Data summary")
print(f" - Parcels: {N}")
print(f" - Functions: {M}")
print(f" - Vacant parcels: {int(vacant_mask.sum())}")
print(f" - Non-vacant parcels: {int((~vacant_mask).sum())}")
print(f" - Max distance: {MAX_DISTANCE} m")
print(f" - Neighbor build time: {elapsed:.2f} sec")

if HAS_PARCEL_GROUND_AREA:
    print(" - parcel_ground_area: available")
else:
    print(" - parcel_ground_area: not available (FAR logic will not work unless this column exists)")

neighbor_counts = np.array([len(x) for x in neighbors_idx], dtype=int)
print(f" - Mean neighbors within radius: {neighbor_counts.mean():.2f}")
print(f" - Max neighbors within radius: {neighbor_counts.max() if len(neighbor_counts) else 0}")

summary_df = pd.DataFrame({
    "parcel_id": parcel_ids[:10],
    "is_vacant": vacant_mask[:10],
    "existing_function_count": existing_function_count[:10],
    "dominant_function": dominant_function[:10],
    "baseline_interparcel_score": baseline_interparcel_scores[:10]
})


In [ ]:
# ============================================================
# CELL 3 — PCM SCENARIO STUDIO  (redesigned UI)
# ============================================================

import json
import re
from copy import deepcopy

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ------------------------------------------------------------
# 0) CHECK REQUIRED OBJECTS FROM CELL 2
# ------------------------------------------------------------
required_objects = [
    "data_bundle", "parcel_ids", "function_cols",
    "parcels_df", "HAS_PARCEL_GROUND_AREA"
]
for obj in required_objects:
    if obj not in globals():
        raise RuntimeError(f"Required object '{obj}' not found. Run Cell 2 first.")

ALL_PARCEL_IDS       = sorted(list(set([str(x).strip() for x in parcel_ids])))
ALL_PARCEL_IDS_SET   = set(ALL_PARCEL_IDS)
ALL_FUNCTIONS        = list(function_cols)
ALL_FUNCTIONS_SET    = set(ALL_FUNCTIONS)

PARCELS_BASE_DF = parcels_df.copy()
PARCELS_BASE_DF["parcel_id"] = PARCELS_BASE_DF["parcel_id"].astype(str).str.strip()
if "parcel_total_functions_area" not in PARCELS_BASE_DF.columns:
    raise RuntimeError("Column 'parcel_total_functions_area' is required in parcels_df.")
PARCEL_ROW_MAP = PARCELS_BASE_DF.set_index("parcel_id").to_dict("index")

# ------------------------------------------------------------
# internal keys
# ------------------------------------------------------------
FUNCTION_DISPLAY_NAMES = {
    "parking":                  "Parking",
    "commercial":               "Commercial",
    "cultural":                 "Cultural",
    "leisure_and_hospitality":  "Leisure & Hospitality",
    "industrial":               "Industrial",
    "sports":                   "Sports",
    "undeveloped_land":         "Unbuilt / Open Space",
    "offices":                  "Offices",
    "unique_building":          "Unique Building",
    "residential":              "Residential",
    "health_and_charity":       "Health & Charity",
    "green":                    "Green Space",
}
FUNCTION_ICONS = {
    "parking":                  "🅿️",
    "commercial":               "🏪",
    "cultural":                 "🎭",
    "leisure_and_hospitality":  "🍽️",
    "industrial":               "🏭",
    "sports":                   "⚽",
    "undeveloped_land":         "🌿",
    "offices":                  "🏢",
    "unique_building":          "🏛️",
    "residential":              "🏠",
    "health_and_charity":       "❤️",
    "green":                    "🌳",
}
def fn_display(key):
    return FUNCTION_DISPLAY_NAMES.get(key, key)

def fn_icon(key):
    return FUNCTION_ICONS.get(key, "")

# ------------------------------------------------------------
# 1) FIXED OPTIMISATION SETTINGS
# ------------------------------------------------------------
FIXED_OPTIMIZATION_SETTINGS = {
    "population_size":   500,
    "n_generations":     500,
    "mutation_rate":     0.15,
    "crossover_rate":    0.80,
    "elite_fraction":    0.10,
    "n_solutions":       10,
    "objective_mode":    "maximize_complementarity"
}

# ------------------------------------------------------------
# 2) STATE INITIALIZATION
# ------------------------------------------------------------
def default_global_settings():
    return {
        "scenario_name": "Scenario 1",
        "notes": "",
        **FIXED_OPTIMIZATION_SETTINGS
    }

def default_group_payload(group_name: str):
    return {
        "group_name":                       group_name,
        "parcel_ids":                       [],
        "allowed_functions":                [],
        "area_mode":                        "bounded_range",
        "group_total_area_min":             None,
        "group_total_area_max":             None,
        "group_area_budget_min":            None,
        "group_area_budget_max":            None,
        "use_far_constraints":              False,
        "far_mode":                         "group",
        "group_far_value":                  None,
        "parcel_far_rules":                 {},
        "use_function_constraints":         False,
        "function_rule_mode":               "group",
        "group_function_rules":             {},
        "parcel_function_rules":            {},
        "min_active_functions":             None,
        "max_active_functions":             None,
        "preserve_existing_functions":      False,
        "allow_new_functions":              True,
        "allow_remove_existing_functions":  True,
        "locked_parcel_ids":                [],
        "function_presence_rules":          {},
        "notes":                            ""
    }

def default_ui_state():
    return {"loaded_group_name": None, "dirty": False, "last_action": None}

if "scenario_config" not in globals():
    scenario_config = {
        "global_settings": default_global_settings(),
        "groups":          {},
        "compiled": {
            "target_parcel_ids":      [],
            "parcel_constraints":     {},
            "validation_summary":     {},
            "compiled_preview_rows":  []
        },
        "ui_state": default_ui_state()
    }
else:
    scenario_config.setdefault("global_settings", default_global_settings())
    scenario_config.setdefault("groups", {})
    scenario_config.setdefault("compiled", {
        "target_parcel_ids": [], "parcel_constraints": {},
        "validation_summary": {}, "compiled_preview_rows": []
    })
    scenario_config.setdefault("ui_state", default_ui_state())

# Undo / redo history (session only)
if "editor_undo_stack" not in globals():
    editor_undo_stack = []   # list of serialised editor snapshots
if "editor_redo_stack" not in globals():
    editor_redo_stack = []
UNDO_MAX_DEPTH = 10

# Trash bin (soft-deleted groups, session only)
if "deleted_groups_trash" not in globals():
    deleted_groups_trash = {}   # {group_name: payload}

# Per-group "last saved" snapshot for one-click revert
if "group_last_saved_snapshot" not in globals():
    group_last_saved_snapshot = {}   # {group_name: payload}

editor_group = default_group_payload("Group 1")

# ------------------------------------------------------------
# 3) HELPERS
# ------------------------------------------------------------
def safe_float_or_none(x):
    if x is None: return None
    if isinstance(x, str) and x.strip() == "": return None
    try:
        if pd.isna(x): return None
    except Exception: pass
    return float(x)

def parse_optional_float(text: str):
    text = str(text).strip()
    if text == "": return None
    val = float(text)
    if not np.isfinite(val): raise ValueError("Value must be finite.")
    return float(val)

def parse_optional_int(text: str):
    text = str(text).strip()
    if text == "": return None
    return int(float(text))

def parse_id_text(text: str):
    if not str(text).strip(): return []
    tokens = re.split(r"[,\n;\t ]+", str(text).strip())
    return list(dict.fromkeys([t.strip() for t in tokens if t.strip()]))

def sanitize_group_name(name: str):
    name = str(name).strip()
    if not name: raise ValueError("Group name cannot be empty.")
    return name

def mark_dirty(flag=True, action=None):
    scenario_config["ui_state"]["dirty"] = bool(flag)
    if action is not None:
        scenario_config["ui_state"]["last_action"] = action
    refresh_status_badge()

def update_global_settings():
    scenario_config["global_settings"] = {
        "scenario_name": scenario_name_text.value.strip() or "Scenario 1",
        "notes":         global_notes_text.value.strip(),
        **FIXED_OPTIMIZATION_SETTINGS
    }

def get_parcel_ground_area(pid: str):
    if not HAS_PARCEL_GROUND_AREA: return None
    return safe_float_or_none(PARCEL_ROW_MAP[pid].get("parcel_ground_area"))

def get_base_total_area(pid: str):
    val = safe_float_or_none(PARCEL_ROW_MAP[pid].get("parcel_total_functions_area"))
    return 0.0 if val is None else float(val)

def get_base_function_row(pid: str):
    row = PARCEL_ROW_MAP[pid]
    return np.array([
        0.0 if safe_float_or_none(row.get(f)) is None else float(row.get(f))
        for f in ALL_FUNCTIONS
    ], dtype=float)

def selected_functions_from_checks(check_dict):
    return [f for f, cb in check_dict.items() if cb.value]

def set_checks_from_list(check_dict, selected_list):
    sel = set(selected_list)
    for f, cb in check_dict.items():
        cb.value = (f in sel)

def clean_rule_dict(rule_dict):
    out = {}
    for f, rule in (rule_dict or {}).items():
        if f not in ALL_FUNCTIONS_SET: continue
        rmin = safe_float_or_none(rule.get("min_area"))
        rmax = safe_float_or_none(rule.get("max_area"))
        if rmin is None and rmax is None: continue
        out[f] = {"min_area": rmin, "max_area": rmax}
    return out

def clean_function_presence_rules(rule_dict):
    out = {}
    for f, v in (rule_dict or {}).items():
        if f not in ALL_FUNCTIONS_SET: continue
        n = parse_optional_int(v) if isinstance(v, str) else (None if v is None else int(v))
        if n is not None and n >= 0: out[f] = int(n)
    return out

def build_parcel_metrics_df(parcel_ids_list):
    rows = []
    for pid in sorted(parcel_ids_list):
        if pid not in PARCEL_ROW_MAP:
            rows.append({"parcel_id": pid, "exists": False, "base_total_area_m²": "—",
                         "ground_area_m²": "—", "base_FAR": "—", "existing_functions": "—"})
            continue
        base_total = get_base_total_area(pid)
        garea      = get_parcel_ground_area(pid)
        base_row   = get_base_function_row(pid)
        active_n   = int(np.sum(base_row > 0))
        base_far   = (base_total / garea) if (garea is not None and garea > 0) else None
        active_fns = [fn_display(f) for f, v in zip(ALL_FUNCTIONS, base_row) if v > 0]
        rows.append({
            "parcel_id":           pid,
            "exists":              True,
            "base_total_area_m²":  f"{base_total:,.0f}",
            "ground_area_m²":      f"{garea:,.0f}" if garea is not None else "—",
            "base_FAR":            f"{base_far:.2f}" if base_far is not None else "—",
            "existing_functions":  ", ".join(active_fns) if active_fns else "—"
        })
    return pd.DataFrame(rows) if rows else pd.DataFrame(
        columns=["parcel_id","exists","base_total_area_m²","ground_area_m²","base_FAR","existing_functions"])

# ------------------------------------------------------------
# UNDO / REDO  helpers
# ------------------------------------------------------------
def _editor_snapshot():
    """Serialise the current editor widget state into a dict."""
    return {
        "group_name":            group_name_text.value,
        "parcel_ids_text":       parcel_editor_text.value,
        "allowed_functions":     [f for f, cb in function_checkboxes.items() if cb.value],
        "area_mode":             area_mode_dropdown.value,
        "group_total_min":       group_total_min_text.value,
        "group_total_max":       group_total_max_text.value,
        "group_budget_min":      group_budget_min_text.value,
        "group_budget_max":      group_budget_max_text.value,
        "use_far":               use_far_checkbox.value,
        "far_mode":              far_mode_dropdown.value,
        "group_far":             group_far_text.value,
        "parcel_far_rules":      deepcopy(editor_group.get("parcel_far_rules", {})),
        "use_fn_constraints":    use_function_constraints_checkbox.value,
        "fn_rule_mode":          function_rule_mode_dropdown.value,
        "group_fn_rules":        deepcopy(editor_group.get("group_function_rules", {})),
        "parcel_fn_rules":       deepcopy(editor_group.get("parcel_function_rules", {})),
        "min_active":            min_active_functions_text.value,
        "max_active":            max_active_functions_text.value,
        "preserve_existing":     preserve_existing_checkbox.value,
        "allow_new":             allow_new_functions_checkbox.value,
        "allow_remove":          allow_remove_existing_checkbox.value,
        "locked_parcels":        locked_parcels_text.value,
        "fn_presence_rules":     deepcopy(editor_group.get("function_presence_rules", {})),
        "notes":                 group_notes_text.value,
    }

def _restore_snapshot(snap):
    """Restore editor widgets from a snapshot dict."""
    global editor_group
    group_name_text.value                   = snap["group_name"]
    parcel_editor_text.value                = snap["parcel_ids_text"]
    set_checks_from_list(function_checkboxes, snap["allowed_functions"])
    area_mode_dropdown.value                = snap["area_mode"]
    group_total_min_text.value              = snap["group_total_min"]
    group_total_max_text.value              = snap["group_total_max"]
    group_budget_min_text.value             = snap["group_budget_min"]
    group_budget_max_text.value             = snap["group_budget_max"]
    use_far_checkbox.value                  = snap["use_far"]
    far_mode_dropdown.value                 = snap["far_mode"]
    group_far_text.value                    = snap["group_far"]
    editor_group["parcel_far_rules"]        = deepcopy(snap["parcel_far_rules"])
    use_function_constraints_checkbox.value = snap["use_fn_constraints"]
    function_rule_mode_dropdown.value       = snap["fn_rule_mode"]
    editor_group["group_function_rules"]    = deepcopy(snap["group_fn_rules"])
    editor_group["parcel_function_rules"]   = deepcopy(snap["parcel_fn_rules"])
    min_active_functions_text.value         = snap["min_active"]
    max_active_functions_text.value         = snap["max_active"]
    preserve_existing_checkbox.value        = snap["preserve_existing"]
    allow_new_functions_checkbox.value      = snap["allow_new"]
    allow_remove_existing_checkbox.value    = snap["allow_remove"]
    locked_parcels_text.value               = snap["locked_parcels"]
    editor_group["function_presence_rules"] = deepcopy(snap["fn_presence_rules"])
    group_notes_text.value                  = snap["notes"]
    refresh_area_mode_visibility()
    refresh_far_visibility()
    refresh_function_rule_visibility()
    render_parcel_metrics()
    render_far_rules_table()
    render_function_rules_table()
    render_function_presence_table()

def push_undo():
    global editor_undo_stack, editor_redo_stack
    editor_undo_stack.append(_editor_snapshot())
    if len(editor_undo_stack) > UNDO_MAX_DEPTH:
        editor_undo_stack.pop(0)
    editor_redo_stack.clear()
    _refresh_undo_redo_buttons()

def do_undo(_=None):
    global editor_undo_stack, editor_redo_stack
    if not editor_undo_stack:
        show_toast("Nothing to undo.", kind="info")
        return
    editor_redo_stack.append(_editor_snapshot())
    snap = editor_undo_stack.pop()
    _restore_snapshot(snap)
    mark_dirty(True, "undo")
    _refresh_undo_redo_buttons()
    show_toast("Undo applied.", kind="success")

def do_redo(_=None):
    global editor_undo_stack, editor_redo_stack
    if not editor_redo_stack:
        show_toast("Nothing to redo.", kind="info")
        return
    editor_undo_stack.append(_editor_snapshot())
    snap = editor_redo_stack.pop()
    _restore_snapshot(snap)
    mark_dirty(True, "redo")
    _refresh_undo_redo_buttons()
    show_toast("Redo applied.", kind="success")

def _refresh_undo_redo_buttons():
    undo_btn.disabled = len(editor_undo_stack) == 0
    redo_btn.disabled = len(editor_redo_stack) == 0
    undo_btn.description = f"↩ Undo ({len(editor_undo_stack)})"
    redo_btn.description = f"↪ Redo ({len(editor_redo_stack)})"

# generic "mark dirty + push undo" observer
_undo_push_pending = False
def on_generic_dirty(change=None):
    global _undo_push_pending
    if not _undo_push_pending:
        _undo_push_pending = True
        push_undo()
        _undo_push_pending = False
    mark_dirty(True, "edit")

# ------------------------------------------------------------
# TOAST notification helper
# ------------------------------------------------------------
def show_toast(msg, kind="success"):
    colours = {
        "success": ("#dcfce7", "#166534", "✅"),
        "error":   ("#fee2e2", "#991b1b", "❌"),
        "warning": ("#fef9c3", "#854d0e", "⚠️"),
        "info":    ("#dbeafe", "#1e40af", "ℹ️"),
    }
    bg, fg, icon = colours.get(kind, colours["info"])
    with toast_out:
        clear_output(wait=True)
        display(widgets.HTML(f"""
        <div style="background:{bg};color:{fg};padding:10px 16px;border-radius:12px;
                    font-size:13px;font-weight:600;margin:6px 0;border-left:4px solid {fg};">
            {icon} {msg}
        </div>"""))

# ------------------------------------------------------------
# 4) BUILD / LOAD EDITOR GROUP
# ------------------------------------------------------------
def build_group_payload_from_editor(validate_now=True):
    payload = default_group_payload(sanitize_group_name(group_name_text.value))
    payload["parcel_ids"]        = sorted(parse_id_text(parcel_editor_text.value))
    payload["allowed_functions"] = selected_functions_from_checks(function_checkboxes)

    payload["area_mode"]             = area_mode_dropdown.value
    payload["group_total_area_min"]  = parse_optional_float(group_total_min_text.value)
    payload["group_total_area_max"]  = parse_optional_float(group_total_max_text.value)
    payload["group_area_budget_min"] = parse_optional_float(group_budget_min_text.value)
    payload["group_area_budget_max"] = parse_optional_float(group_budget_max_text.value)

    payload["use_far_constraints"] = bool(use_far_checkbox.value)
    payload["far_mode"]            = far_mode_dropdown.value
    payload["group_far_value"]     = parse_optional_float(group_far_text.value)
    payload["parcel_far_rules"]    = deepcopy(editor_group.get("parcel_far_rules", {}))

    payload["use_function_constraints"] = bool(use_function_constraints_checkbox.value)
    payload["function_rule_mode"]       = function_rule_mode_dropdown.value
    payload["group_function_rules"]     = clean_rule_dict(deepcopy(editor_group.get("group_function_rules", {})))
    payload["parcel_function_rules"]    = {
        pid: clean_rule_dict(rules)
        for pid, rules in deepcopy(editor_group.get("parcel_function_rules", {})).items()
        if clean_rule_dict(rules)
    }

    min_active = parse_optional_int(min_active_functions_text.value)
    max_active = parse_optional_int(max_active_functions_text.value)
    payload["min_active_functions"] = min_active
    payload["max_active_functions"] = max_active

    payload["preserve_existing_functions"]      = bool(preserve_existing_checkbox.value)
    payload["allow_new_functions"]              = bool(allow_new_functions_checkbox.value)
    payload["allow_remove_existing_functions"]  = bool(allow_remove_existing_checkbox.value)
    payload["locked_parcel_ids"]                = sorted(parse_id_text(locked_parcels_text.value))
    payload["function_presence_rules"]          = clean_function_presence_rules(
                                                      deepcopy(editor_group.get("function_presence_rules", {})))
    payload["notes"] = group_notes_text.value.strip()

    if validate_now:
        validate_group_payload(payload)
    return payload

def load_group_to_editor(group_name):
    global editor_group, editor_undo_stack, editor_redo_stack
    if group_name not in scenario_config["groups"]:
        raise ValueError(f"Group '{group_name}' not found.")

    editor_group = deepcopy(scenario_config["groups"][group_name])
    editor_undo_stack.clear()
    editor_redo_stack.clear()
    _refresh_undo_redo_buttons()

    group_name_text.value               = editor_group["group_name"]
    parcel_editor_text.value            = "\n".join(editor_group.get("parcel_ids", []))
    set_checks_from_list(function_checkboxes, editor_group.get("allowed_functions", []))

    area_mode_dropdown.value            = editor_group.get("area_mode", "bounded_range")
    group_total_min_text.value          = "" if editor_group.get("group_total_area_min") is None else str(editor_group["group_total_area_min"])
    group_total_max_text.value          = "" if editor_group.get("group_total_area_max") is None else str(editor_group["group_total_area_max"])
    group_budget_min_text.value         = "" if editor_group.get("group_area_budget_min") is None else str(editor_group["group_area_budget_min"])
    group_budget_max_text.value         = "" if editor_group.get("group_area_budget_max") is None else str(editor_group["group_area_budget_max"])

    use_far_checkbox.value              = bool(editor_group.get("use_far_constraints", False))
    far_mode_dropdown.value             = editor_group.get("far_mode", "group")
    group_far_text.value                = "" if editor_group.get("group_far_value") is None else str(editor_group["group_far_value"])
    far_parcel_id_text.value            = ""
    far_value_text.value                = ""

    use_function_constraints_checkbox.value = bool(editor_group.get("use_function_constraints", False))
    function_rule_mode_dropdown.value       = editor_group.get("function_rule_mode", "group")
    group_function_min_text.value           = ""
    group_function_max_text.value           = ""
    parcel_function_min_text.value          = ""
    parcel_function_max_text.value          = ""
    pf_parcel_id_text.value                 = ""

    min_active_functions_text.value     = "" if editor_group.get("min_active_functions") is None else str(editor_group["min_active_functions"])
    max_active_functions_text.value     = "" if editor_group.get("max_active_functions") is None else str(editor_group["max_active_functions"])
    preserve_existing_checkbox.value    = bool(editor_group.get("preserve_existing_functions", False))
    allow_new_functions_checkbox.value  = bool(editor_group.get("allow_new_functions", True))
    allow_remove_existing_checkbox.value= bool(editor_group.get("allow_remove_existing_functions", True))
    locked_parcels_text.value           = "\n".join(editor_group.get("locked_parcel_ids", []))
    fp_function_required_count_text.value = ""
    group_notes_text.value              = editor_group.get("notes", "")

    scenario_config["ui_state"]["loaded_group_name"] = group_name
    scenario_config["ui_state"]["dirty"]             = False
    scenario_config["ui_state"]["last_action"]       = "load"

    refresh_area_mode_visibility()
    refresh_far_visibility()
    refresh_function_rule_visibility()
    refresh_status_badge()
    render_parcel_metrics()
    render_far_rules_table()
    render_function_rules_table()
    render_function_presence_table()
    render_saved_groups_panel()
    render_compiled_preview()
    render_diagnostics()

def reset_editor(new_group_name=None):
    global editor_group, editor_undo_stack, editor_redo_stack
    editor_group = default_group_payload(new_group_name or f"Group {len(scenario_config['groups']) + 1}")
    editor_undo_stack.clear()
    editor_redo_stack.clear()
    _refresh_undo_redo_buttons()

    group_name_text.value               = editor_group["group_name"]
    parcel_editor_text.value            = ""
    set_checks_from_list(function_checkboxes, [])
    area_mode_dropdown.value            = "bounded_range"
    group_total_min_text.value          = ""
    group_total_max_text.value          = ""
    group_budget_min_text.value         = ""
    group_budget_max_text.value         = ""
    use_far_checkbox.value              = False
    far_mode_dropdown.value             = "group"
    group_far_text.value                = ""
    far_parcel_id_text.value            = ""
    far_value_text.value                = ""
    editor_group["parcel_far_rules"]    = {}
    use_function_constraints_checkbox.value  = False
    function_rule_mode_dropdown.value        = "group"
    group_function_min_text.value       = ""
    group_function_max_text.value       = ""
    parcel_function_min_text.value      = ""
    parcel_function_max_text.value      = ""
    pf_parcel_id_text.value             = ""
    editor_group["group_function_rules"]     = {}
    editor_group["parcel_function_rules"]    = {}
    min_active_functions_text.value     = ""
    max_active_functions_text.value     = ""
    preserve_existing_checkbox.value    = False
    allow_new_functions_checkbox.value  = True
    allow_remove_existing_checkbox.value= True
    locked_parcels_text.value           = ""
    editor_group["function_presence_rules"]  = {}
    fp_function_required_count_text.value = ""
    group_notes_text.value              = ""

    scenario_config["ui_state"]["loaded_group_name"] = None
    scenario_config["ui_state"]["dirty"]             = False
    scenario_config["ui_state"]["last_action"]       = "reset"

    refresh_area_mode_visibility()
    refresh_far_visibility()
    refresh_function_rule_visibility()
    refresh_status_badge()
    render_parcel_metrics()
    render_far_rules_table()
    render_function_rules_table()
    render_function_presence_table()

# ------------------------------------------------------------
# 5) VALIDATION
# ------------------------------------------------------------
def validate_group_payload(payload):
    gname = payload["group_name"]
    if len(payload["parcel_ids"]) == 0:
        raise ValueError(f"Group '{gname}' must contain at least one parcel.")
    bad_parcels = [p for p in payload["parcel_ids"] if p not in ALL_PARCEL_IDS_SET]
    if bad_parcels:
        raise ValueError(f"Unknown parcel IDs in '{gname}': {bad_parcels[:10]}")
    bad_allowed = [f for f in payload["allowed_functions"] if f not in ALL_FUNCTIONS_SET]
    if bad_allowed:
        raise ValueError(f"Unknown allowed functions in '{gname}': {bad_allowed}")
    if len(payload["allowed_functions"]) == 0:
        raise ValueError(f"Group '{gname}' must have at least one allowed function.")
    area_mode = payload.get("area_mode", "bounded_range")
    if area_mode not in ("bounded_range", "flexible"):
        raise ValueError(f"Group '{gname}': invalid area mode.")
    tmin = safe_float_or_none(payload.get("group_total_area_min"))
    tmax = safe_float_or_none(payload.get("group_total_area_max"))
    if tmin is not None and tmin < 0: raise ValueError(f"Group '{gname}': total area min cannot be negative.")
    if tmax is not None and tmax < 0: raise ValueError(f"Group '{gname}': total area max cannot be negative.")
    if tmin is not None and tmax is not None and tmin > tmax:
        raise ValueError(f"Group '{gname}': total area min cannot exceed total area max.")
    bmin = safe_float_or_none(payload.get("group_area_budget_min"))
    bmax = safe_float_or_none(payload.get("group_area_budget_max"))
    if bmin is not None and bmin < 0: raise ValueError(f"Group '{gname}': area budget min cannot be negative.")
    if bmax is not None and bmax < 0: raise ValueError(f"Group '{gname}': area budget max cannot be negative.")
    if bmin is not None and bmax is not None and bmin > bmax:
        raise ValueError(f"Group '{gname}': area budget min cannot exceed area budget max.")
    if payload["use_far_constraints"]:
        if not HAS_PARCEL_GROUND_AREA:
            raise ValueError("FAR constraints cannot be used because 'parcel_ground_area' is not available.")
        if payload["far_mode"] == "group":
            fv = payload["group_far_value"]
            if fv is None: raise ValueError(f"Group '{gname}': enter one FAR value.")
            if fv < 0:     raise ValueError(f"Group '{gname}': FAR cannot be negative.")
        elif payload["far_mode"] == "parcel_specific":
            if len(payload["parcel_far_rules"]) == 0:
                raise ValueError(f"Group '{gname}': add at least one parcel-specific FAR rule.")
            for pid, fv in payload["parcel_far_rules"].items():
                if pid not in payload["parcel_ids"]:
                    raise ValueError(f"Group '{gname}': parcel FAR rule uses parcel '{pid}' not in group.")
                if fv is None or fv < 0:
                    raise ValueError(f"Group '{gname}' / {pid}: FAR must be non-negative.")
        else:
            raise ValueError(f"Group '{gname}': invalid FAR mode.")
    if payload["use_function_constraints"]:
        mode = payload["function_rule_mode"]
        if mode not in ("group", "parcel_specific", "both"):
            raise ValueError(f"Group '{gname}': invalid function rule mode.")
        if mode in ("group", "both"):
            for f, rule in payload["group_function_rules"].items():
                rmin = safe_float_or_none(rule.get("min_area"))
                rmax = safe_float_or_none(rule.get("max_area"))
                if rmin is not None and rmin < 0: raise ValueError(f"{gname}/{f}: min area cannot be negative.")
                if rmax is not None and rmax < 0: raise ValueError(f"{gname}/{f}: max area cannot be negative.")
                if rmin is not None and rmax is not None and rmin > rmax:
                    raise ValueError(f"{gname}/{f}: min area cannot exceed max area.")
        if mode in ("parcel_specific", "both"):
            for pid, fd in payload["parcel_function_rules"].items():
                if pid not in payload["parcel_ids"]:
                    raise ValueError(f"Group '{gname}': parcel rule uses parcel '{pid}' not in group.")
                for f, rule in fd.items():
                    rmin = safe_float_or_none(rule.get("min_area"))
                    rmax = safe_float_or_none(rule.get("max_area"))
                    if rmin is not None and rmin < 0: raise ValueError(f"{gname}/{pid}/{f}: min area cannot be negative.")
                    if rmax is not None and rmax < 0: raise ValueError(f"{gname}/{pid}/{f}: max area cannot be negative.")
                    if rmin is not None and rmax is not None and rmin > rmax:
                        raise ValueError(f"{gname}/{pid}/{f}: min area cannot exceed max area.")
    min_active = payload.get("min_active_functions")
    max_active = payload.get("max_active_functions")
    if min_active is not None and min_active < 0: raise ValueError(f"Group '{gname}': min active functions cannot be negative.")
    if max_active is not None and max_active < 0: raise ValueError(f"Group '{gname}': max active functions cannot be negative.")
    if min_active is not None and max_active is not None and min_active > max_active:
        raise ValueError(f"Group '{gname}': min active cannot exceed max active functions.")
    if max_active is not None and max_active > len(payload["allowed_functions"]):
        raise ValueError(f"Group '{gname}': max active cannot exceed number of allowed functions.")
    bad_locked = [p for p in payload.get("locked_parcel_ids", []) if p not in payload["parcel_ids"]]
    if bad_locked:
        raise ValueError(f"Group '{gname}': locked parcel IDs not in group: {bad_locked[:10]}")

# ------------------------------------------------------------
# 6) COMPILE
# ------------------------------------------------------------
def compile_group_to_parcel_constraints(group_payload):
    gname            = group_payload["group_name"]
    allowed_functions= list(group_payload["allowed_functions"])
    allowed_set      = set(allowed_functions)
    disallowed_functions = [f for f in ALL_FUNCTIONS if f not in allowed_set]

    use_far          = bool(group_payload.get("use_far_constraints", False))
    far_mode         = group_payload.get("far_mode", "group")
    group_far_value  = safe_float_or_none(group_payload.get("group_far_value"))
    parcel_far_rules = group_payload.get("parcel_far_rules", {}) or {}

    use_function_constraints = bool(group_payload.get("use_function_constraints", False))
    function_rule_mode       = group_payload.get("function_rule_mode", "group")
    group_function_rules     = clean_rule_dict(group_payload.get("group_function_rules", {}) or {})
    parcel_function_rules    = group_payload.get("parcel_function_rules", {}) or {}

    area_mode        = group_payload.get("area_mode", "fixed_current")
    min_active       = group_payload.get("min_active_functions")
    max_active       = group_payload.get("max_active_functions")
    preserve_existing= bool(group_payload.get("preserve_existing_functions", False))
    allow_new        = bool(group_payload.get("allow_new_functions", True))
    allow_remove_existing = bool(group_payload.get("allow_remove_existing_functions", True))
    locked_set       = set(group_payload.get("locked_parcel_ids", []) or [])
    function_presence_rules = clean_function_presence_rules(group_payload.get("function_presence_rules", {}) or {})

    compiled     = {}
    preview_rows = []
    errors       = []
    warnings     = []

    for pid in group_payload["parcel_ids"]:
        parcel_errors   = []
        parcel_warnings = []

        if pid not in PARCEL_ROW_MAP:
            parcel_errors.append(f"{pid}: parcel not found in dataset.")
            compiled[pid] = {"parcel_id": pid, "group_name": gname, "feasible": False,
                             "errors": parcel_errors, "warnings": parcel_warnings}
            errors.extend(parcel_errors)
            continue

        ground_area    = get_parcel_ground_area(pid)
        base_total_area= get_base_total_area(pid)
        base_row       = get_base_function_row(pid)
        base_active_functions = [ALL_FUNCTIONS[j] for j in range(len(ALL_FUNCTIONS)) if base_row[j] > 0]

        effective_far          = None
        total_area_max_from_far= None
        if use_far:
            if far_mode == "group" and group_far_value is not None:
                effective_far = group_far_value
            elif far_mode == "parcel_specific" and pid in parcel_far_rules:
                effective_far = parcel_far_rules[pid]
            if effective_far is not None and ground_area is not None and ground_area > 0:
                total_area_max_from_far = effective_far * ground_area
            elif effective_far is not None:
                parcel_warnings.append(f"{pid}: FAR set but ground area missing or zero.")

        if area_mode in ("bounded_range", "fixed_current"):
            # fixed_current kept for backwards compatibility with older saved configs
            total_area_min = safe_float_or_none(group_payload.get("group_total_area_min"))
            total_area_max = safe_float_or_none(group_payload.get("group_total_area_max"))
            if total_area_max_from_far is not None:
                if total_area_max is None:
                    total_area_max = total_area_max_from_far
                else:
                    total_area_max = min(total_area_max, total_area_max_from_far)
        else:
            # flexible: optimiser decides, only budget cap applies
            total_area_min = None
            total_area_max = total_area_max_from_far

        function_bounds = {}
        for f in ALL_FUNCTIONS:
            if f not in allowed_set:
                function_bounds[f] = {"min_area": 0.0, "max_area": 0.0}
                continue
            rmin, rmax = None, None
            if use_function_constraints:
                if function_rule_mode in ("group", "both") and f in group_function_rules:
                    rmin = safe_float_or_none(group_function_rules[f].get("min_area"))
                    rmax = safe_float_or_none(group_function_rules[f].get("max_area"))
                pf_rules = clean_rule_dict(parcel_function_rules.get(pid, {}))
                if function_rule_mode in ("parcel_specific", "both") and f in pf_rules:
                    rmin = safe_float_or_none(pf_rules[f].get("min_area"))
                    rmax = safe_float_or_none(pf_rules[f].get("max_area"))
            function_bounds[f] = {"min_area": rmin, "max_area": rmax}

        for f, rule in function_bounds.items():
            rmin = safe_float_or_none(rule.get("min_area"))
            rmax = safe_float_or_none(rule.get("max_area"))
            if rmin is None: rmin = 0.0
            if rmin < 0:  parcel_errors.append(f"{pid}/{f}: min area cannot be negative.")
            if rmax is not None and rmax < 0: parcel_errors.append(f"{pid}/{f}: max area cannot be negative.")
            if rmax is not None and rmin > rmax: parcel_errors.append(f"{pid}/{f}: min area exceeds max area.")
            function_bounds[f]["min_area"] = float(rmin)
            function_bounds[f]["max_area"] = None if rmax is None else float(rmax)

        if total_area_max is not None:
            for f in ALL_FUNCTIONS:
                rmax = function_bounds[f]["max_area"]
                if rmax is not None:
                    function_bounds[f]["max_area"] = min(float(rmax), float(total_area_max))

        for f, rule in function_bounds.items():
            rmin = rule["min_area"]; rmax = rule["max_area"]
            if rmax is not None and rmin > rmax:
                parcel_errors.append(f"{pid}/{f}: min exceeds effective max after total caps.")

        min_sum = float(sum(rule["min_area"] for rule in function_bounds.values()))
        positive_capacity_functions = []
        constrained_functions_count = 0
        for f, rule in function_bounds.items():
            rmin = rule["min_area"]; rmax = rule["max_area"]
            if rmax is None or rmax > 0: positive_capacity_functions.append(f)
            if (rmin not in (None, 0.0)) or (rmax is not None): constrained_functions_count += 1

        if total_area_max is not None and min_sum > total_area_max + 1e-9:
            parcel_errors.append(f"{pid}: sum of min function areas ({min_sum:.2f}) exceeds total cap ({total_area_max:.2f}).")
        if total_area_min is not None and total_area_max is not None and total_area_min > total_area_max:
            parcel_errors.append(f"{pid}: total area min exceeds total area max.")
        if len(positive_capacity_functions) == 0:
            parcel_errors.append(f"{pid}: no function has positive capacity after constraints.")
        allowed_active_existing_count = len([f for f in base_active_functions if f in allowed_set])
        if preserve_existing and allowed_active_existing_count == 0:
            parcel_warnings.append(f"{pid}: preserve-existing is active but parcel has no existing allowed functions.")
        if min_active is not None and min_active > len(positive_capacity_functions):
            parcel_errors.append(f"{pid}: min active functions ({min_active}) exceeds functions with positive capacity ({len(positive_capacity_functions)}).")
        if max_active is not None and max_active > len(allowed_functions):
            parcel_errors.append(f"{pid}: max active functions ({max_active}) exceeds number of allowed functions ({len(allowed_functions)}).")

        is_locked = pid in locked_set
        if is_locked:
            total_area_min = float(base_total_area)
            total_area_max = float(base_total_area)
            function_bounds = {}
            for j, f in enumerate(ALL_FUNCTIONS):
                val = float(base_row[j])
                function_bounds[f] = {"min_area": val, "max_area": val}

        feasible = (len(parcel_errors) == 0)
        compiled[pid] = {
            "parcel_id": pid, "group_name": gname,
            "allowed_functions": allowed_functions, "disallowed_functions": disallowed_functions,
            "ground_area": ground_area, "base_total_functions_area": base_total_area,
            "effective_far": effective_far, "total_area_max_from_far": total_area_max_from_far,
            "area_mode": area_mode, "total_area_min": total_area_min, "total_area_max": total_area_max,
            "group_budget_min": safe_float_or_none(group_payload.get("group_area_budget_min")),
            "group_budget_max": safe_float_or_none(group_payload.get("group_area_budget_max")),
            "function_bounds": function_bounds,
            "min_active_functions": min_active, "max_active_functions": max_active,
            "preserve_existing_functions": preserve_existing,
            "allow_new_functions": allow_new,
            "allow_remove_existing_functions": allow_remove_existing,
            "locked": is_locked, "function_presence_rules": function_presence_rules,
            "feasible": feasible, "errors": parcel_errors, "warnings": parcel_warnings
        }
        preview_rows.append({
            "parcel_id": pid, "group_name": gname, "locked": is_locked,
            "ground_area": ground_area, "effective_far": effective_far, "area_mode": area_mode,
            "total_area_min": total_area_min, "total_area_max": total_area_max,
            "allowed_functions_count": len(allowed_functions),
            "positive_capacity_functions": len(positive_capacity_functions),
            "constrained_functions_count": constrained_functions_count,
            "min_active_functions": min_active, "max_active_functions": max_active,
            "feasible": feasible, "n_warnings": len(parcel_warnings), "n_errors": len(parcel_errors)
        })
        errors.extend(parcel_errors)
        warnings.extend(parcel_warnings)

    for f, req_n in function_presence_rules.items():
        if req_n > len(group_payload["parcel_ids"]):
            warnings.append(f"{gname}: required presence count for '{f}' ({req_n}) exceeds number of parcels.")

    budget_min = safe_float_or_none(group_payload.get("group_area_budget_min"))
    budget_max = safe_float_or_none(group_payload.get("group_area_budget_max"))
    if budget_min is not None or budget_max is not None:
        sum_min = 0.0; sum_max = 0.0; max_unbounded = False
        for pid in group_payload["parcel_ids"]:
            c = compiled[pid]
            sum_min += float(c.get("total_area_min") or 0.0)
            if c.get("total_area_max") is None: max_unbounded = True
            else: sum_max += float(c["total_area_max"])
        if budget_max is not None and sum_min > budget_max:
            errors.append(f"{gname}: sum of parcel min totals ({sum_min:.2f}) exceeds group budget max ({budget_max:.2f}).")
        if budget_min is not None and (not max_unbounded) and sum_max < budget_min:
            errors.append(f"{gname}: sum of parcel max totals ({sum_max:.2f}) is below group budget min ({budget_min:.2f}).")

    return compiled, preview_rows, errors, warnings

def compile_full_scenario():
    all_groups         = scenario_config.get("groups", {})
    parcel_constraints = {}
    preview_rows       = []
    all_errors         = []
    all_warnings       = []
    overwritten_parcels= []

    for group_name in sorted(all_groups.keys()):
        group_payload = deepcopy(all_groups[group_name])
        validate_group_payload(group_payload)
        compiled_group, preview_group, group_errors, group_warnings = compile_group_to_parcel_constraints(group_payload)
        for pid, c in compiled_group.items():
            if pid in parcel_constraints: overwritten_parcels.append(pid)
            parcel_constraints[pid] = c
        preview_rows.extend(preview_group)
        all_errors.extend(group_errors)
        all_warnings.extend(group_warnings)

    target_parcel_ids = sorted(parcel_constraints.keys())
    feasible_n  = int(sum(1 for pid in target_parcel_ids if parcel_constraints[pid]["feasible"]))
    infeasible_n= len(target_parcel_ids) - feasible_n

    validation_summary = {
        "n_groups":                          len(all_groups),
        "n_target_parcels":                  len(target_parcel_ids),
        "n_feasible_parcels":                feasible_n,
        "n_infeasible_parcels":              infeasible_n,
        "overwritten_parcels_last_group_wins": sorted(list(set(overwritten_parcels))),
        "errors":                            all_errors,
        "warnings":                          all_warnings
    }
    scenario_config["compiled"] = {
        "target_parcel_ids":     target_parcel_ids,
        "parcel_constraints":    parcel_constraints,
        "validation_summary":    validation_summary,
        "compiled_preview_rows": preview_rows
    }
    return scenario_config["compiled"]

def validate_current_scenario():
    update_global_settings()
    return compile_full_scenario()

# ============================================================
# UI — WIDGETS
# ============================================================

# ---- shared style ----
W_FULL = widgets.Layout(width="100%")
W_WIDE = widgets.Layout(width="420px")
W_MED  = widgets.Layout(width="280px")
W_SM   = widgets.Layout(width="180px")
W_XS   = widgets.Layout(width="120px")
BTN_SM = widgets.Layout(width="110px", height="34px")
BTN_MD = widgets.Layout(width="140px", height="34px")

toast_out = widgets.Output()

# ---- STEP 1: Scenario Setup ----
scenario_name_text = widgets.Text(
    value=scenario_config["global_settings"]["scenario_name"],
    placeholder="e.g. Scenario A – Mixed Use",
    layout=W_WIDE
)
global_notes_text = widgets.Textarea(
    value=scenario_config["global_settings"]["notes"],
    placeholder="Optional: describe the planning intent of this scenario…",
    layout=widgets.Layout(width="100%", height="80px")
)
validate_scenario_btn = widgets.Button(
    description="✅ Validate & Compile", button_style="success", layout=BTN_MD
)
status_badge_out = widgets.Output()

def refresh_status_badge():
    loaded = scenario_config["ui_state"].get("loaded_group_name")
    dirty  = scenario_config["ui_state"].get("dirty", False)
    vs     = scenario_config.get("compiled", {}).get("validation_summary", {})
    n_g    = vs.get("n_groups", 0)
    n_p    = vs.get("n_target_parcels", 0)
    n_ok   = vs.get("n_feasible_parcels", 0)
    n_err  = len(vs.get("errors", []))
    n_warn = len(vs.get("warnings", []))

    dirty_html = (
        '<span style="color:#b45309;font-weight:700;">⚠ Unsaved changes</span>'
        if dirty else
        '<span style="color:#166534;font-weight:700;">✓ Saved</span>'
    )
    err_html = (
        f'<span style="color:#991b1b;font-weight:700;">{n_err} error{"s" if n_err!=1 else ""}</span>'
        if n_err else
        f'<span style="color:#166534;">{n_err} errors</span>'
    )
    warn_html = (
        f'<span style="color:#854d0e;font-weight:700;">{n_warn} warning{"s" if n_warn!=1 else ""}</span>'
        if n_warn else
        f'<span style="color:#166534;">{n_warn} warnings</span>'
    )
    with status_badge_out:
        clear_output(wait=True)
        display(widgets.HTML(f"""
        <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:center;
                    background:#f1f5f9;border-radius:12px;padding:10px 16px;
                    border:1px solid #e2e8f0;font-size:13px;margin-top:4px;">
          <span>📋 <b>{n_g}</b> group{"s" if n_g!=1 else ""}</span>
          <span>🗺️ <b>{n_p}</b> parcel{"s" if n_p!=1 else ""}</span>
          <span>✅ <b>{n_ok}</b> feasible</span>
          <span>{err_html}</span>
          <span>{warn_html}</span>
          <span style="margin-left:auto;">{dirty_html}</span>
          <span style="color:#475569;">Loaded: <b>{loaded or "—"}</b></span>
        </div>"""))

# ---- STEP 2: Group Manager ----
saved_group_dropdown = widgets.Dropdown(
    options=["— select a group —"] + sorted(list(scenario_config["groups"].keys())),
    value="— select a group —",
    layout=W_WIDE
)
load_group_btn      = widgets.Button(description="📂 Load",          layout=BTN_SM, button_style="success")
new_group_btn       = widgets.Button(description="➕ New",            layout=BTN_SM, button_style="info")
save_group_btn      = widgets.Button(description="💾 Save",           layout=BTN_SM, button_style="primary")
save_as_new_btn     = widgets.Button(description="💾 Save as New",    layout=BTN_MD)
# bottom save bar buttons (same handlers, different widget instances)
save_group_btn_bottom   = widgets.Button(description="💾 Save this group",  button_style="primary", layout=widgets.Layout(width="160px", height="42px"))
save_as_new_btn_bottom  = widgets.Button(description="💾 Save as New",      layout=widgets.Layout(width="150px", height="42px"))
reset_editor_btn_bottom = widgets.Button(description="↺ Start over",        button_style="warning",  layout=widgets.Layout(width="130px", height="42px"))
toast_out_bottom        = widgets.Output()
duplicate_group_btn = widgets.Button(description="⧉ Duplicate",      layout=BTN_SM)
delete_group_btn    = widgets.Button(description="🗑 Delete",          layout=BTN_SM, button_style="danger")
revert_group_btn    = widgets.Button(description="⏪ Revert to saved", layout=BTN_MD)
restore_trash_btn   = widgets.Button(description="♻ Restore deleted", layout=BTN_MD)

undo_btn = widgets.Button(description="↩ Undo (0)", layout=widgets.Layout(width="130px", height="34px"), disabled=True)
redo_btn = widgets.Button(description="↪ Redo (0)", layout=widgets.Layout(width="130px", height="34px"), disabled=True)

group_name_text = widgets.Text(
    value=editor_group["group_name"],
    placeholder="Name this group, e.g. 'City Centre'",
    layout=W_WIDE
)

# confirm-delete overlay
_delete_confirm_pending = False
delete_confirm_yes_btn = widgets.Button(description="⚠ Yes, delete",  button_style="danger", layout=BTN_MD)
delete_confirm_no_btn  = widgets.Button(description="✗ Cancel",        layout=BTN_SM)
delete_confirm_box     = widgets.HBox([
    widgets.HTML('<span style="font-size:13px;font-weight:700;color:#991b1b;padding:0 10px;">Delete this group?</span>'),
    delete_confirm_yes_btn, delete_confirm_no_btn
], layout=widgets.Layout(display="none"))

trash_dropdown_out = widgets.Output()

# ---- STEP 2b: Parcel Editor ----
parcel_editor_text = widgets.Textarea(
    value="",
    placeholder="Paste parcel IDs here — one per line, or separated by commas or spaces.",
    layout=widgets.Layout(width="100%", height="130px")
)
parcel_sort_btn   = widgets.Button(description="⇅ Sort",            layout=BTN_SM)
parcel_dedupe_btn = widgets.Button(description="⊘ Remove duplicates", layout=BTN_MD)
parcel_clear_btn  = widgets.Button(description="✕ Clear all",       layout=BTN_SM, button_style="danger")
parcel_count_out  = widgets.Output()
parcel_metrics_out= widgets.Output()

# ---- STEP 2c: Functions ----
function_checkboxes = {
    f: widgets.Checkbox(value=False, description=f"{fn_icon(f)}  {fn_display(f)}",
                        indent=False, layout=widgets.Layout(width="210px"))
    for f in ALL_FUNCTIONS
}
select_all_functions_btn= widgets.Button(description="✔ Select all", layout=BTN_SM)
clear_all_functions_btn = widgets.Button(description="✕ Clear all",  layout=BTN_SM)

# build 3-column grid
_fn_rows = []
_fn_row  = []
for _i, _f in enumerate(ALL_FUNCTIONS, start=1):
    _fn_row.append(function_checkboxes[_f])
    if _i % 3 == 0:
        _fn_rows.append(widgets.HBox(_fn_row))
        _fn_row = []
if _fn_row:
    _fn_rows.append(widgets.HBox(_fn_row))

# ---- STEP 3: Constraints ----
#  3a area behavior
area_mode_dropdown = widgets.Dropdown(
    options=[
        ("Set a development range  (recommended)",  "bounded_range"),
        ("Let the optimiser decide — flexible",     "flexible"),
    ],
    value="bounded_range", layout=W_FULL
)
group_total_min_text  = widgets.Text(value="", placeholder="e.g. 5000", description="Min m²:", layout=W_SM)
group_total_max_text  = widgets.Text(value="", placeholder="e.g. 12000", description="Max m²:", layout=W_SM)
group_budget_min_text = widgets.Text(value="", placeholder="optional",   description="Budget min m²:", layout=W_SM)
group_budget_max_text = widgets.Text(value="", placeholder="optional",   description="Budget max m²:", layout=W_SM)
area_bounds_box       = widgets.HBox([group_total_min_text, group_total_max_text],
                                      layout=widgets.Layout(margin="6px 0"))
budget_box            = widgets.HBox([group_budget_min_text, group_budget_max_text],
                                      layout=widgets.Layout(margin="2px 0"))

#  3b FAR
use_far_checkbox   = widgets.Checkbox(value=False, description="Enable FAR (Floor-Area Ratio) constraints",
                                       indent=False, layout=widgets.Layout(width="340px"))
far_mode_dropdown  = widgets.Dropdown(
    options=[("One FAR value for the whole group", "group"),
             ("Different FAR per parcel",           "parcel_specific")],
    value="group", layout=W_MED
)
group_far_text     = widgets.Text(value="", placeholder="e.g. 2.5", description="FAR:", layout=W_SM)
far_parcel_id_text = widgets.Text(value="", placeholder="Parcel ID", description="Parcel:", layout=W_SM)
far_value_text     = widgets.Text(value="", placeholder="e.g. 1.8",  description="FAR:",    layout=W_XS)
far_add_btn        = widgets.Button(description="+ Add / Update", button_style="primary", layout=BTN_MD)
far_remove_btn     = widgets.Button(description="− Remove",       button_style="warning",  layout=BTN_SM)
far_rules_out      = widgets.Output()
far_mode_box       = widgets.HBox([far_mode_dropdown])
group_far_box      = widgets.HBox([group_far_text])
parcel_far_box     = widgets.HBox([far_parcel_id_text, far_value_text, far_add_btn, far_remove_btn])

#  3c function rules
use_function_constraints_checkbox = widgets.Checkbox(
    value=False, description="Enable min / max area rules per function",
    indent=False, layout=widgets.Layout(width="360px"))
function_rule_mode_dropdown = widgets.Dropdown(
    options=[("Rules apply to the whole group",  "group"),
             ("Rules apply per parcel",          "parcel_specific"),
             ("Both group and parcel rules",     "both")],
    value="group", layout=W_MED
)
group_function_dropdown    = widgets.Dropdown(options=[(fn_icon(f) + "  " + fn_display(f), f) for f in ALL_FUNCTIONS], layout=W_MED,
    description="Function:")
group_function_min_text    = widgets.Text(value="", placeholder="e.g. 500",  description="Min m²:", layout=W_SM)
group_function_max_text    = widgets.Text(value="", placeholder="e.g. 3000", description="Max m²:", layout=W_SM)
group_function_add_btn     = widgets.Button(description="+ Add / Update", button_style="primary", layout=BTN_MD)
group_function_remove_btn  = widgets.Button(description="− Remove",       button_style="warning",  layout=BTN_SM)
pf_parcel_id_text          = widgets.Text(value="", placeholder="Parcel ID", description="Parcel:", layout=W_SM)
pf_function_dropdown       = widgets.Dropdown(options=[(fn_icon(f) + "  " + fn_display(f), f) for f in ALL_FUNCTIONS], layout=W_MED,
    description="Function:")
parcel_function_min_text   = widgets.Text(value="", placeholder="e.g. 200",  description="Min m²:", layout=W_SM)
parcel_function_max_text   = widgets.Text(value="", placeholder="e.g. 1500", description="Max m²:", layout=W_SM)
parcel_function_add_btn    = widgets.Button(description="+ Add / Update", button_style="primary", layout=BTN_MD)
parcel_function_remove_btn = widgets.Button(description="− Remove",       button_style="warning",  layout=BTN_SM)
function_rules_out         = widgets.Output()
function_mode_box          = widgets.HBox([function_rule_mode_dropdown])
group_function_rule_box    = widgets.VBox([
    widgets.HTML('<div style="font-size:12px;color:#475569;margin:4px 0;">Group-level rule:</div>'),
    widgets.HBox([group_function_dropdown, group_function_min_text,
                  group_function_max_text, group_function_add_btn, group_function_remove_btn])
])
parcel_function_rule_box   = widgets.VBox([
    widgets.HTML('<div style="font-size:12px;color:#475569;margin:4px 0;">Parcel-level rule:</div>'),
    widgets.HBox([pf_parcel_id_text, pf_function_dropdown, parcel_function_min_text,
                  parcel_function_max_text, parcel_function_add_btn, parcel_function_remove_btn])
])

#  3d advanced
min_active_functions_text    = widgets.Text(value="", placeholder="e.g. 2", description="Min active:", layout=W_SM)
max_active_functions_text    = widgets.Text(value="", placeholder="e.g. 5", description="Max active:", layout=W_SM)
preserve_existing_checkbox   = widgets.Checkbox(value=False,
    description="Keep existing functions (don't remove them)", indent=False,
    layout=widgets.Layout(width="330px"))
allow_new_functions_checkbox = widgets.Checkbox(value=True,
    description="Allow new functions to be added", indent=False,
    layout=widgets.Layout(width="280px"))
allow_remove_existing_checkbox = widgets.Checkbox(value=True,
    description="Allow existing functions to be removed", indent=False,
    layout=widgets.Layout(width="330px"))
locked_parcels_text = widgets.Textarea(value="",
    placeholder="Optional: paste parcel IDs that must stay exactly as-is",
    layout=widgets.Layout(width="100%", height="80px"))
fp_function_dropdown           = widgets.Dropdown(options=[(fn_icon(f) + "  " + fn_display(f), f) for f in ALL_FUNCTIONS], layout=W_MED,
    description="Function:")
fp_function_required_count_text= widgets.Text(value="", placeholder="e.g. 3",
    description="Min parcels:", layout=W_SM)
fp_add_btn    = widgets.Button(description="+ Add / Update", button_style="primary", layout=BTN_MD)
fp_remove_btn = widgets.Button(description="− Remove",       button_style="warning",  layout=BTN_SM)
function_presence_out = widgets.Output()
group_notes_text = widgets.Textarea(value="",
    placeholder="Optional notes about this group's planning intent…",
    layout=widgets.Layout(width="100%", height="70px"))

# ---- STEP 4: Review ----
saved_groups_out    = widgets.Output()
compiled_preview_out= widgets.Output()
diagnostics_out     = widgets.Output()
launch_out          = widgets.Output()

# ============================================================
# VISIBILITY helpers
# ============================================================
def refresh_area_mode_visibility(*args):
    mode = area_mode_dropdown.value
    # Min/Max range fields: only for bounded_range
    area_bounds_box.layout.display = "flex" if mode == "bounded_range" else "none"
    # Budget cap: useful for both modes
    budget_box.layout.display      = "flex"

def refresh_far_visibility(*args):
    active = use_far_checkbox.value
    mode   = far_mode_dropdown.value
    far_mode_box.layout.display   = "flex" if active else "none"
    group_far_box.layout.display  = "flex" if (active and mode == "group") else "none"
    parcel_far_box.layout.display = "flex" if (active and mode == "parcel_specific") else "none"
    far_rules_out.layout.display  = "block" if (active and mode == "parcel_specific") else "none"

def refresh_function_rule_visibility(*args):
    function_mode_box.layout.display         = "flex" if use_function_constraints_checkbox.value else "none"
    group_function_rule_box.layout.display   = "flex" if (use_function_constraints_checkbox.value and
        function_rule_mode_dropdown.value in ("group", "both")) else "none"
    parcel_function_rule_box.layout.display  = "flex" if (use_function_constraints_checkbox.value and
        function_rule_mode_dropdown.value in ("parcel_specific", "both")) else "none"

# ============================================================
# RENDER helpers
# ============================================================
def render_parcel_metrics():
    pids = parse_id_text(parcel_editor_text.value)
    with parcel_count_out:
        clear_output(wait=True)
        color = "#166534" if pids else "#6b7280"
        display(widgets.HTML(
            f'<span style="font-size:13px;font-weight:600;color:{color};">'
            f'{"" if not pids else f"{len(pids)} parcel ID(s) entered"}'
            f'</span>'
        ))
    with parcel_metrics_out:
        clear_output(wait=True)
        if not pids:
            display(widgets.HTML('<p style="color:#94a3b8;font-size:13px;">No parcel IDs entered yet.</p>'))
            return
        df = build_parcel_metrics_df(pids)
        bad = df.loc[df["exists"] == False, "parcel_id"].tolist() if "exists" in df.columns else []
        if bad:
            display(widgets.HTML(
                f'<div style="background:#fee2e2;color:#991b1b;padding:8px 12px;'
                f'border-radius:8px;font-size:12px;margin-bottom:6px;">'
                f'⚠ These IDs were not found in the dataset: {", ".join(bad[:10])}'
                f'{"…" if len(bad)>10 else ""}</div>'
            ))
        display(df.drop(columns=["exists"], errors="ignore").head(200))
        if len(df) > 200:
            display(widgets.HTML(f'<p style="color:#64748b;font-size:12px;">…showing first 200 of {len(df)}</p>'))

def render_far_rules_table():
    with far_rules_out:
        clear_output(wait=True)
        rules = editor_group.get("parcel_far_rules", {})
        if not rules:
            display(widgets.HTML('<p style="color:#94a3b8;font-size:13px;">No parcel-specific FAR rules added.</p>'))
            return
        df = pd.DataFrame([{"Parcel ID": pid, "FAR ratio": v} for pid, v in sorted(rules.items())])
        display(df)

def render_function_rules_table():
    with function_rules_out:
        clear_output(wait=True)
        rows = []
        for f, r in sorted(clean_rule_dict(editor_group.get("group_function_rules", {})).items()):
            rows.append({"Scope": "Group", "Parcel": "—", "Function": fn_display(f),
                         "Min m²": r.get("min_area","—"), "Max m²": r.get("max_area","—")})
        for pid, fd in sorted(editor_group.get("parcel_function_rules", {}).items()):
            for f, r in sorted(clean_rule_dict(fd).items()):
                rows.append({"Scope": "Parcel", "Parcel": pid, "Function": fn_display(f),
                             "Min m²": r.get("min_area","—"), "Max m²": r.get("max_area","—")})
        if not rows:
            display(widgets.HTML('<p style="color:#94a3b8;font-size:13px;">No function area rules added yet.</p>'))
            return
        display(pd.DataFrame(rows))

def render_function_presence_table():
    with function_presence_out:
        clear_output(wait=True)
        rules = editor_group.get("function_presence_rules", {})
        if not rules:
            display(widgets.HTML('<p style="color:#94a3b8;font-size:13px;">No presence requirements added.</p>'))
            return
        df = pd.DataFrame([{"Function": fn_display(f), "Required in at least N parcels": n}
                           for f, n in sorted(rules.items())])
        display(df)

def render_saved_groups_panel():
    with saved_groups_out:
        clear_output(wait=True)
        groups = scenario_config["groups"]
        trash  = deleted_groups_trash

        if not groups and not trash:
            display(widgets.HTML('<p style="color:#94a3b8;font-size:13px;">No groups saved yet.</p>'))
            return

        cards_html = ""
        for name, g in sorted(groups.items()):
            n_p   = len(g.get("parcel_ids", []))
            n_fn  = len(g.get("allowed_functions", []))
            fns   = ", ".join(fn_display(f) for f in g.get("allowed_functions", []))
            mode  = g.get("area_mode", "—").replace("_", " ")
            notes = g.get("notes", "")
            has_snap = name in group_last_saved_snapshot
            snap_tag = '<span style="color:#0369a1;font-size:11px;">⏪ revertable</span>' if has_snap else ""
            cards_html += f"""
            <div style="border:1px solid #e2e8f0;border-radius:12px;padding:14px 16px;
                        margin-bottom:10px;background:#f8fafc;">
              <div style="display:flex;justify-content:space-between;align-items:center;">
                <span style="font-weight:700;font-size:14px;color:#0f172a;">📦 {name}</span>
                {snap_tag}
              </div>
              <div style="font-size:12px;color:#475569;margin-top:6px;">
                🗺️ <b>{n_p}</b> parcels &nbsp;|&nbsp;
                🏗️ <b>{n_fn}</b> functions &nbsp;|&nbsp;
                📐 area mode: <b>{mode}</b>
              </div>
              <div style="font-size:12px;color:#64748b;margin-top:4px;line-height:1.5;">
                {fns if fns else "<i>no functions selected</i>"}
              </div>
              {"<div style='font-size:12px;color:#94a3b8;margin-top:4px;font-style:italic;'>" + notes + "</div>" if notes else ""}
            </div>"""

        if trash:
            cards_html += '<div style="margin-top:12px;"><b style="font-size:13px;color:#991b1b;">🗑 Trash (this session)</b></div>'
            for name in sorted(trash.keys()):
                cards_html += f"""
                <div style="border:1px dashed #fca5a5;border-radius:10px;padding:10px 14px;
                            margin-top:6px;background:#fff5f5;opacity:0.85;">
                  <span style="font-size:13px;color:#991b1b;">🗑 {name}</span>
                  <span style="font-size:11px;color:#94a3b8;margin-left:8px;">(deleted this session)</span>
                </div>"""

        display(widgets.HTML(cards_html))

def render_compiled_preview():
    with compiled_preview_out:
        clear_output(wait=True)
        try:
            compiled = compile_full_scenario()
            rows = compiled["compiled_preview_rows"]
            if not rows:
                display(widgets.HTML('<p style="color:#94a3b8;font-size:13px;">Save at least one group first.</p>'))
                return
            df = pd.DataFrame(rows).sort_values(["group_name", "parcel_id"]).reset_index(drop=True)
            # friendly column names
            df = df.rename(columns={
                "parcel_id": "Parcel ID", "group_name": "Group",
                "locked": "Locked", "ground_area": "Ground m²",
                "effective_far": "FAR", "area_mode": "Area Mode",
                "total_area_min": "Min m²", "total_area_max": "Max m²",
                "allowed_functions_count": "Allowed fns",
                "positive_capacity_functions": "Viable fns",
                "min_active_functions": "Min active", "max_active_functions": "Max active",
                "feasible": "Feasible", "n_warnings": "Warnings", "n_errors": "Errors"
            })
            display(df.head(300))
            if len(df) > 300:
                display(widgets.HTML(f'<p style="color:#64748b;font-size:12px;">…showing first 300 of {len(df)}</p>'))
        except Exception as e:
            display(widgets.HTML(
                f'<div style="background:#fee2e2;color:#991b1b;padding:8px 12px;border-radius:8px;">'
                f'⚠ Compilation error: {e}</div>'
            ))

def render_diagnostics():
    with diagnostics_out:
        clear_output(wait=True)
        try:
            compiled = compile_full_scenario()
            vs = compiled.get("validation_summary", {})
            errors   = vs.get("errors",   [])
            warnings = vs.get("warnings", [])
            overwritten = vs.get("overwritten_parcels_last_group_wins", [])

            n_g   = vs.get("n_groups", 0)
            n_p   = vs.get("n_target_parcels", 0)
            n_ok  = vs.get("n_feasible_parcels", 0)
            n_bad = vs.get("n_infeasible_parcels", 0)

            overall_ok = (len(errors) == 0 and n_bad == 0)
            banner_bg  = "#dcfce7" if overall_ok else "#fee2e2"
            banner_fg  = "#166534" if overall_ok else "#991b1b"
            banner_msg = "✅ All checks passed — ready for Cell 4!" if overall_ok else f"❌ {len(errors)} error(s) found — fix before running Cell 4."

            html = f"""
            <div style="border-radius:14px;overflow:hidden;border:1px solid #e2e8f0;">
              <div style="background:{banner_bg};color:{banner_fg};padding:14px 18px;
                          font-size:15px;font-weight:700;">{banner_msg}</div>
              <div style="padding:14px 18px;background:#f8fafc;display:flex;gap:20px;flex-wrap:wrap;font-size:13px;">
                <span>📦 <b>{n_g}</b> group{"s" if n_g!=1 else ""}</span>
                <span>🗺️ <b>{n_p}</b> parcel{"s" if n_p!=1 else ""} targeted</span>
                <span style="color:#166534;">✅ <b>{n_ok}</b> feasible</span>
                <span style="color:{"#991b1b" if n_bad else "#166534"};">{"❌" if n_bad else "✅"} <b>{n_bad}</b> infeasible</span>
                <span style="color:{"#854d0e" if warnings else "#166534"};">⚠ <b>{len(warnings)}</b> warning{"s" if len(warnings)!=1 else ""}</span>
                <span style="color:{"#991b1b" if errors else "#166534"};">🚫 <b>{len(errors)}</b> error{"s" if len(errors)!=1 else ""}</span>
              </div>
            </div>"""
            display(widgets.HTML(html))

            if errors:
                display(widgets.HTML('<div style="margin-top:10px;font-weight:700;color:#991b1b;font-size:13px;">❌ Errors</div>'))
                for e in errors[:30]:
                    display(widgets.HTML(f'<div style="font-size:12px;color:#991b1b;padding:3px 0;">• {e}</div>'))
                if len(errors) > 30:
                    display(widgets.HTML(f'<div style="font-size:12px;color:#94a3b8;">…and {len(errors)-30} more</div>'))

            if warnings:
                display(widgets.HTML('<div style="margin-top:8px;font-weight:700;color:#854d0e;font-size:13px;">⚠ Warnings</div>'))
                for w in warnings[:20]:
                    display(widgets.HTML(f'<div style="font-size:12px;color:#854d0e;padding:3px 0;">• {w}</div>'))
                if len(warnings) > 20:
                    display(widgets.HTML(f'<div style="font-size:12px;color:#94a3b8;">…and {len(warnings)-20} more</div>'))

            if overwritten:
                display(widgets.HTML(
                    f'<div style="margin-top:8px;font-size:12px;color:#0369a1;">'
                    f'ℹ️ Parcels assigned to multiple groups (last group wins): {", ".join(overwritten[:20])}</div>'
                ))

        except Exception as e:
            display(widgets.HTML(
                f'<div style="background:#fee2e2;color:#991b1b;padding:8px 12px;border-radius:8px;">'
                f'⚠ Diagnostic error: {e}</div>'
            ))

def render_launch_panel():
    with launch_out:
        clear_output(wait=True)
        compiled = scenario_config.get("compiled", {})
        vs = compiled.get("validation_summary", {})
        n_p  = vs.get("n_target_parcels", 0)
        n_ok = vs.get("n_feasible_parcels", 0)
        n_err= len(vs.get("errors", []))
        ready= (n_err == 0 and n_p > 0 and n_ok == n_p)
        bg   = "#f0fdf4" if ready else "#fff7ed"
        fg   = "#166534" if ready else "#9a3412"
        icon = "🚀" if ready else "⏳"
        msg  = (f"Scenario is ready! Run <b>Cell 4</b> to start the optimisation for {n_p} parcels."
                if ready else
                f"Scenario is not ready yet. Fix {n_err} error(s) and ensure all parcels are feasible.")
        display(widgets.HTML(f"""
        <div style="background:{bg};border:2px solid {fg};border-radius:16px;
                    padding:20px 24px;text-align:center;">
          <div style="font-size:28px;margin-bottom:8px;">{icon}</div>
          <div style="font-size:15px;font-weight:700;color:{fg};">{msg}</div>
          <div style="font-size:13px;color:#475569;margin-top:10px;">
            When ready, go to <b>Cell 4</b> and run it. The <code>scenario_config</code>
            variable is automatically passed — no copy-paste needed.
          </div>
        </div>"""))

# ============================================================
# REFRESH DROPDOWN
# ============================================================
def refresh_saved_group_dropdown():
    current = saved_group_dropdown.value
    opts    = ["— select a group —"] + sorted(list(scenario_config["groups"].keys()))
    saved_group_dropdown.options = opts
    if current in opts:
        saved_group_dropdown.value = current
    else:
        saved_group_dropdown.value = "— select a group —"

def render_trash_dropdown():
    with trash_dropdown_out:
        clear_output(wait=True)
        if not deleted_groups_trash:
            display(widgets.HTML('<p style="color:#94a3b8;font-size:12px;">Trash is empty.</p>'))
            return
        opts = list(sorted(deleted_groups_trash.keys()))
        dd   = widgets.Dropdown(options=opts, layout=W_MED, description="Restore:")
        btn  = widgets.Button(description="♻ Restore", button_style="warning", layout=BTN_SM)
        def _restore(b):
            name = dd.value
            if name and name in deleted_groups_trash:
                scenario_config["groups"][name] = deepcopy(deleted_groups_trash[name])
                del deleted_groups_trash[name]
                compile_full_scenario()
                refresh_saved_group_dropdown()
                render_saved_groups_panel()
                render_diagnostics()
                render_trash_dropdown()
                render_launch_panel()
                refresh_status_badge()
                show_toast(f"Group '{name}' restored from trash.", kind="success")
        btn.on_click(_restore)
        display(widgets.HBox([dd, btn]))

# ============================================================
# EVENT HANDLERS
# ============================================================
def on_validate_scenario(_):
    try:
        update_global_settings()
        compile_full_scenario()
        render_diagnostics()
        render_compiled_preview()
        render_saved_groups_panel()
        render_launch_panel()
        refresh_status_badge()
        show_toast("Scenario validated and compiled.", kind="success")
    except Exception as e:
        show_toast(f"Validation error: {e}", kind="error")

def on_load_group(_):
    dirty = scenario_config["ui_state"].get("dirty", False)
    selected = saved_group_dropdown.value
    if selected == "— select a group —":
        show_toast("Select a group first.", kind="warning"); return

    if dirty:
        with toast_out:
            clear_output(wait=True)
            display(widgets.HTML(
                f'<div style="background:#fef9c3;color:#854d0e;padding:10px 16px;border-radius:12px;'
                f'font-size:13px;font-weight:600;border-left:4px solid #ca8a04;">'
                f'⚠ You have unsaved changes. '
                f'<span style="text-decoration:underline;cursor:pointer;" '
                f'id="load_anyway_yes">Load anyway and discard?</span>'
                f'</div>'))
        _pending_load[0] = selected
        confirm_load_btn.layout.display  = "inline-flex"
        cancel_load_btn.layout.display   = "inline-flex"
        return

    _do_load(selected)

_pending_load = [None]
confirm_load_btn = widgets.Button(description="Yes, discard & load", button_style="warning",
                                   layout=widgets.Layout(display="none", width="170px", height="34px"))
cancel_load_btn  = widgets.Button(description="Cancel",               layout=widgets.Layout(display="none",
                                   width="90px", height="34px"))

def _do_load(name):
    try:
        load_group_to_editor(name)
        saved_group_dropdown.value = name
        confirm_load_btn.layout.display = "none"
        cancel_load_btn.layout.display  = "none"
        render_launch_panel()
        show_toast(f"Group '{name}' loaded.", kind="success")
    except Exception as e:
        show_toast(f"Load error: {e}", kind="error")

def on_confirm_load(_):
    name = _pending_load[0]
    if name: _do_load(name)

def on_cancel_load(_):
    _pending_load[0] = None
    confirm_load_btn.layout.display = "none"
    cancel_load_btn.layout.display  = "none"
    show_toast("Load cancelled.", kind="info")

def on_new_group(_):
    global editor_undo_stack, editor_redo_stack
    reset_editor(new_group_name=f"Group {len(scenario_config['groups']) + 1}")
    render_launch_panel()
    show_toast("New empty group ready. Fill in details and save.", kind="info")

def on_save_group(_):
    global editor_group
    try:
        update_global_settings()
        payload = build_group_payload_from_editor(validate_now=True)
        name    = payload["group_name"]
        # store last-saved snapshot before overwriting
        if name in scenario_config["groups"]:
            group_last_saved_snapshot[name] = deepcopy(scenario_config["groups"][name])
        scenario_config["groups"][name]                  = deepcopy(payload)
        editor_group                                     = deepcopy(payload)
        scenario_config["ui_state"]["loaded_group_name"] = name
        scenario_config["ui_state"]["dirty"]             = False
        scenario_config["ui_state"]["last_action"]       = "save"
        compile_full_scenario()
        refresh_saved_group_dropdown()
        saved_group_dropdown.value = name
        render_saved_groups_panel()
        render_diagnostics()
        render_compiled_preview()
        render_launch_panel()
        refresh_status_badge()
        show_toast(f"Group '{name}' saved ✓", kind="success")
    except Exception as e:
        show_toast(f"Save error: {e}", kind="error")

def on_save_as_new(_):
    global editor_group
    try:
        update_global_settings()
        payload  = build_group_payload_from_editor(validate_now=True)
        new_name = sanitize_group_name(group_name_text.value)
        if new_name in scenario_config["groups"]:
            show_toast(f"A group named '{new_name}' already exists. Change the name first.", kind="warning"); return
        scenario_config["groups"][new_name]              = deepcopy(payload)
        editor_group                                     = deepcopy(payload)
        scenario_config["ui_state"]["loaded_group_name"] = new_name
        scenario_config["ui_state"]["dirty"]             = False
        scenario_config["ui_state"]["last_action"]       = "save_as_new"
        compile_full_scenario()
        refresh_saved_group_dropdown()
        saved_group_dropdown.value = new_name
        render_saved_groups_panel()
        render_diagnostics()
        render_compiled_preview()
        render_launch_panel()
        refresh_status_badge()
        show_toast(f"Saved as new group: '{new_name}'", kind="success")
    except Exception as e:
        show_toast(f"Error: {e}", kind="error")

def on_duplicate_group(_):
    selected = saved_group_dropdown.value
    if selected == "— select a group —":
        show_toast("Select a group to duplicate.", kind="warning"); return
    base     = deepcopy(scenario_config["groups"][selected])
    new_name = f"{selected}_copy"
    counter  = 2
    while new_name in scenario_config["groups"]:
        new_name = f"{selected}_copy_{counter}"; counter += 1
    base["group_name"]              = new_name
    scenario_config["groups"][new_name] = deepcopy(base)
    compile_full_scenario()
    refresh_saved_group_dropdown()
    saved_group_dropdown.value = new_name
    load_group_to_editor(new_name)
    scenario_config["ui_state"]["dirty"]       = False
    scenario_config["ui_state"]["last_action"] = "duplicate"
    render_saved_groups_panel()
    render_diagnostics()
    render_launch_panel()
    refresh_status_badge()
    show_toast(f"Duplicated as '{new_name}'", kind="success")

def on_delete_group_request(_):
    selected = saved_group_dropdown.value
    if selected == "— select a group —":
        show_toast("Select a group to delete.", kind="warning"); return
    delete_confirm_box.layout.display = "flex"

def on_delete_confirm_yes(_):
    global deleted_groups_trash
    selected = saved_group_dropdown.value
    delete_confirm_box.layout.display = "none"
    if selected == "— select a group —" or selected not in scenario_config["groups"]:
        show_toast("Group not found.", kind="warning"); return
    deleted_groups_trash[selected] = deepcopy(scenario_config["groups"][selected])
    del scenario_config["groups"][selected]
    if selected in group_last_saved_snapshot:
        del group_last_saved_snapshot[selected]
    compile_full_scenario()
    refresh_saved_group_dropdown()
    reset_editor(new_group_name=f"Group {len(scenario_config['groups']) + 1}")
    render_saved_groups_panel()
    render_diagnostics()
    render_launch_panel()
    render_trash_dropdown()
    refresh_status_badge()
    show_toast(f"Group '{selected}' moved to trash. You can restore it below.", kind="warning")

def on_delete_confirm_no(_):
    delete_confirm_box.layout.display = "none"
    show_toast("Delete cancelled.", kind="info")

def on_revert_group(_):
    loaded = scenario_config["ui_state"].get("loaded_group_name")
    if not loaded or loaded not in group_last_saved_snapshot:
        show_toast("No previous save to revert to for this group.", kind="warning"); return
    scenario_config["groups"][loaded] = deepcopy(group_last_saved_snapshot[loaded])
    del group_last_saved_snapshot[loaded]
    load_group_to_editor(loaded)
    compile_full_scenario()
    render_saved_groups_panel()
    render_diagnostics()
    render_launch_panel()
    refresh_status_badge()
    show_toast(f"Reverted '{loaded}' to previous save.", kind="success")

def on_parcel_sort(_):
    push_undo()
    ids = sorted(parse_id_text(parcel_editor_text.value))
    parcel_editor_text.value = "\n".join(ids)
    mark_dirty(True, "parcel_sort")
    render_parcel_metrics()
    show_toast("Parcel IDs sorted.", kind="info")

def on_parcel_dedupe(_):
    push_undo()
    ids = parse_id_text(parcel_editor_text.value)
    parcel_editor_text.value = "\n".join(ids)
    mark_dirty(True, "parcel_dedupe")
    render_parcel_metrics()
    show_toast("Duplicate IDs removed.", kind="info")

def on_parcel_clear(_):
    push_undo()
    parcel_editor_text.value = ""
    mark_dirty(True, "parcel_clear")
    render_parcel_metrics()
    show_toast("Parcel list cleared.", kind="warning")

def on_select_all_functions(_):
    push_undo()
    for cb in function_checkboxes.values(): cb.value = True
    mark_dirty(True, "fn_select_all")

def on_clear_all_functions(_):
    push_undo()
    for cb in function_checkboxes.values(): cb.value = False
    mark_dirty(True, "fn_clear_all")

def on_add_update_far_rule(_):
    try:
        pid  = far_parcel_id_text.value.strip()
        fv   = parse_optional_float(far_value_text.value)
        pids = parse_id_text(parcel_editor_text.value)
        if pid == "":
            raise ValueError("Enter a parcel ID.")
        if pid not in ALL_PARCEL_IDS_SET:
            raise ValueError(f"Parcel ID '{pid}' does not exist in the dataset.")
        if pids and pid not in pids:
            raise ValueError(
                f"Parcel '{pid}' is not in this group's parcel list. "
                f"Add it in Step 2 → Parcel IDs first, then save the group."
            )
        if not pids:
            raise ValueError(
                "No parcel IDs are entered yet. "
                "Go to Step 2 → Parcel IDs, add your parcels first."
            )
        if fv is None:
            raise ValueError("Enter a FAR ratio value (e.g. 2.5).")
        if fv < 0:
            raise ValueError("FAR cannot be negative.")
        push_undo()
        editor_group.setdefault("parcel_far_rules", {})[pid] = fv
        mark_dirty(True, "far_add")
        render_far_rules_table()
        show_toast(f"FAR rule saved for parcel {pid} (FAR = {fv}).", kind="success")
    except Exception as e:
        show_toast(str(e), kind="error")

def on_remove_far_rule(_):
    pid = far_parcel_id_text.value.strip()
    if pid in editor_group.get("parcel_far_rules", {}):
        push_undo()
        del editor_group["parcel_far_rules"][pid]
        mark_dirty(True, "far_remove")
        render_far_rules_table()
        show_toast(f"FAR rule removed for {pid}.", kind="info")
    else:
        show_toast("No FAR rule found for that parcel ID.", kind="warning")

def on_add_update_group_function_rule(_):
    try:
        f    = group_function_dropdown.value
        rmin = parse_optional_float(group_function_min_text.value)
        rmax = parse_optional_float(group_function_max_text.value)
        if rmin is not None and rmin < 0: raise ValueError("Min area cannot be negative.")
        if rmax is not None and rmax < 0: raise ValueError("Max area cannot be negative.")
        if rmin is not None and rmax is not None and rmin > rmax: raise ValueError("Min cannot exceed max.")
        push_undo()
        editor_group.setdefault("group_function_rules", {})[f] = {"min_area": rmin, "max_area": rmax}
        mark_dirty(True, "gfn_add")
        render_function_rules_table()
        show_toast(f"Rule saved for {fn_display(f)}.", kind="success")
    except Exception as e:
        show_toast(str(e), kind="error")

def on_remove_group_function_rule(_):
    f = group_function_dropdown.value
    if f in editor_group.get("group_function_rules", {}):
        push_undo()
        del editor_group["group_function_rules"][f]
        mark_dirty(True, "gfn_remove")
        render_function_rules_table()
        show_toast(f"Rule removed for {fn_display(f)}.", kind="info")
    else:
        show_toast("No rule found for that function.", kind="warning")

def on_add_update_parcel_function_rule(_):
    try:
        pid  = pf_parcel_id_text.value.strip()
        f    = pf_function_dropdown.value
        rmin = parse_optional_float(parcel_function_min_text.value)
        rmax = parse_optional_float(parcel_function_max_text.value)
        pids = parse_id_text(parcel_editor_text.value)
        if pid == "":          raise ValueError("Enter a parcel ID.")
        if pid not in pids:    raise ValueError("Parcel ID not in current group.")
        if rmin is not None and rmin < 0: raise ValueError("Min area cannot be negative.")
        if rmax is not None and rmax < 0: raise ValueError("Max area cannot be negative.")
        if rmin is not None and rmax is not None and rmin > rmax: raise ValueError("Min cannot exceed max.")
        push_undo()
        editor_group.setdefault("parcel_function_rules", {}).setdefault(pid, {})[f] = {"min_area": rmin, "max_area": rmax}
        mark_dirty(True, "pfn_add")
        render_function_rules_table()
        show_toast(f"Rule saved for {pid} / {fn_display(f)}.", kind="success")
    except Exception as e:
        show_toast(str(e), kind="error")

def on_remove_parcel_function_rule(_):
    pid = pf_parcel_id_text.value.strip()
    f   = pf_function_dropdown.value
    if pid in editor_group.get("parcel_function_rules", {}) and f in editor_group["parcel_function_rules"][pid]:
        push_undo()
        del editor_group["parcel_function_rules"][pid][f]
        if not editor_group["parcel_function_rules"][pid]:
            del editor_group["parcel_function_rules"][pid]
        mark_dirty(True, "pfn_remove")
        render_function_rules_table()
        show_toast(f"Rule removed for {pid} / {fn_display(f)}.", kind="info")
    else:
        show_toast("No rule found.", kind="warning")

def on_add_update_function_presence_rule(_):
    try:
        f  = fp_function_dropdown.value
        n  = parse_optional_int(fp_function_required_count_text.value)
        if n is None:  raise ValueError("Enter a required parcel count.")
        if n < 0:      raise ValueError("Count cannot be negative.")
        push_undo()
        editor_group.setdefault("function_presence_rules", {})[f] = int(n)
        mark_dirty(True, "fp_add")
        render_function_presence_table()
        show_toast(f"Presence rule saved for {fn_display(f)}.", kind="success")
    except Exception as e:
        show_toast(str(e), kind="error")

def on_remove_function_presence_rule(_):
    f = fp_function_dropdown.value
    if f in editor_group.get("function_presence_rules", {}):
        push_undo()
        del editor_group["function_presence_rules"][f]
        mark_dirty(True, "fp_remove")
        render_function_presence_table()
        show_toast(f"Presence rule removed for {fn_display(f)}.", kind="info")
    else:
        show_toast("No rule found.", kind="warning")

# ============================================================
# WIRE EVENTS
# ============================================================
validate_scenario_btn.on_click(on_validate_scenario)
load_group_btn.on_click(on_load_group)
confirm_load_btn.on_click(on_confirm_load)
cancel_load_btn.on_click(on_cancel_load)
new_group_btn.on_click(on_new_group)
save_group_btn.on_click(on_save_group)
save_as_new_btn.on_click(on_save_as_new)
save_group_btn_bottom.on_click(on_save_group)
save_as_new_btn_bottom.on_click(on_save_as_new)
reset_editor_btn_bottom.on_click(on_new_group)
duplicate_group_btn.on_click(on_duplicate_group)
delete_group_btn.on_click(on_delete_group_request)
delete_confirm_yes_btn.on_click(on_delete_confirm_yes)
delete_confirm_no_btn.on_click(on_delete_confirm_no)
revert_group_btn.on_click(on_revert_group)
undo_btn.on_click(do_undo)
redo_btn.on_click(do_redo)
parcel_sort_btn.on_click(on_parcel_sort)
parcel_dedupe_btn.on_click(on_parcel_dedupe)
parcel_clear_btn.on_click(on_parcel_clear)
select_all_functions_btn.on_click(on_select_all_functions)
clear_all_functions_btn.on_click(on_clear_all_functions)
far_add_btn.on_click(on_add_update_far_rule)
far_remove_btn.on_click(on_remove_far_rule)
group_function_add_btn.on_click(on_add_update_group_function_rule)
group_function_remove_btn.on_click(on_remove_group_function_rule)
parcel_function_add_btn.on_click(on_add_update_parcel_function_rule)
parcel_function_remove_btn.on_click(on_remove_parcel_function_rule)
fp_add_btn.on_click(on_add_update_function_presence_rule)
fp_remove_btn.on_click(on_remove_function_presence_rule)

area_mode_dropdown.observe(refresh_area_mode_visibility, names="value")
use_far_checkbox.observe(refresh_far_visibility, names="value")
far_mode_dropdown.observe(refresh_far_visibility, names="value")
use_function_constraints_checkbox.observe(refresh_function_rule_visibility, names="value")
function_rule_mode_dropdown.observe(refresh_function_rule_visibility, names="value")

# parcel textarea — auto-refresh metrics on change
parcel_editor_text.observe(lambda c: render_parcel_metrics(), names="value")

for _w in [
    scenario_name_text, global_notes_text, group_name_text, parcel_editor_text,
    group_total_min_text, group_total_max_text, group_budget_min_text, group_budget_max_text,
    group_far_text, min_active_functions_text, max_active_functions_text,
    locked_parcels_text, group_notes_text,
    preserve_existing_checkbox, allow_new_functions_checkbox, allow_remove_existing_checkbox,
    use_far_checkbox, use_function_constraints_checkbox, far_mode_dropdown,
    area_mode_dropdown, function_rule_mode_dropdown
]:
    try: _w.observe(on_generic_dirty, names="value")
    except Exception: pass

for _cb in function_checkboxes.values():
    _cb.observe(on_generic_dirty, names="value")

# ============================================================
# LAYOUT — ACCORDION STEPS + SECTION CARDS
# ============================================================

def _accordion_step(step_num, step_label, title, subtitle, body_children, accent="#3b82f6", open_by_default=True):
    """
    Step card with a clickable header button that shows / hides the body.
    Uses layout.display toggle — no Accordion widget needed.
    """
    state = {"open": open_by_default}

    toggle_btn = widgets.Button(
        description=f"  Step {step_num}  ·  {title}   {'▲ Hide' if open_by_default else '▼ Show'}",
        layout=widgets.Layout(width="100%", height="56px"),
        style={"button_color": "#0f172a", "font_weight": "800"}
    )

    step_label_widget = widgets.HTML(f"""
    <div style="background:linear-gradient(135deg,#0f172a 0%,#1e3a5f 60%,#1e40af 100%);
                color:white;padding:14px 20px 0 20px;border-radius:14px 14px 0 0;
                border:1px solid rgba(255,255,255,0.07);border-bottom:none;">
      <div style="display:flex;align-items:center;gap:14px;">
        <div style="width:32px;height:32px;border-radius:50%;background:{accent};
                    display:flex;align-items:center;justify-content:center;
                    font-weight:800;font-size:14px;color:white;flex-shrink:0;">
          {step_num}
        </div>
        <div style="flex:1;">
          <div style="font-size:11px;opacity:0.7;letter-spacing:1.5px;text-transform:uppercase;">{step_label}</div>
          <div style="font-size:16px;font-weight:800;margin-top:1px;">{title}</div>
          <div style="font-size:12px;opacity:0.85;margin-top:3px;">{subtitle}</div>
        </div>
      </div>
    </div>""")

    hint_bar_open   = widgets.HTML(f"""
    <div style="background:#0f172a;color:white;padding:6px 20px 10px 20px;
                border-radius:0;border-left:1px solid rgba(255,255,255,0.07);
                border-right:1px solid rgba(255,255,255,0.07);
                font-size:12px;cursor:pointer;text-align:right;opacity:0.65;">
      ▲ Click to hide this step
    </div>""")

    hint_bar_closed = widgets.HTML(f"""
    <div style="background:#0f172a;color:white;padding:6px 20px 10px 20px;
                border-radius:0 0 14px 14px;
                border:1px solid rgba(255,255,255,0.07);border-top:none;
                font-size:12px;cursor:pointer;text-align:right;opacity:0.65;">
      ▼ Click to show this step
    </div>""")

    body = widgets.VBox(body_children, layout=widgets.Layout(
        padding="18px 20px", width="100%",
        border="1px solid #dbe4f0",
        border_radius="0",
        background_color="#fafcff",
        display="flex" if open_by_default else "none"
    ))

    hint_bar_open.layout.display   = "block" if open_by_default else "none"
    hint_bar_closed.layout.display = "none"  if open_by_default else "block"

    def _toggle(b):
        state["open"] = not state["open"]
        if state["open"]:
            body.layout.display           = "flex"
            hint_bar_open.layout.display  = "block"
            hint_bar_closed.layout.display= "none"
        else:
            body.layout.display           = "none"
            hint_bar_open.layout.display  = "none"
            hint_bar_closed.layout.display= "block"

    # make the whole header area clickable via a transparent overlay button
    header_btn = widgets.Button(
        description="",
        layout=widgets.Layout(width="100%", height="10px", margin="0", padding="0"),
        style={"button_color": "transparent"}
    )
    header_btn.on_click(_toggle)
    hint_bar_open._click_handler  = _toggle
    hint_bar_closed._click_handler= _toggle

    # Wire a proper visible toggle button in the hint bar
    toggle_open_btn  = widgets.Button(description="▲  Hide this step", layout=widgets.Layout(width="160px", height="30px"))
    toggle_close_btn = widgets.Button(description="▼  Show this step", layout=widgets.Layout(width="160px", height="30px"))
    toggle_open_btn.layout.display  = "inline-flex" if open_by_default else "none"
    toggle_close_btn.layout.display = "none" if open_by_default else "inline-flex"

    def _toggle2(b):
        state["open"] = not state["open"]
        if state["open"]:
            body.layout.display              = "flex"
            toggle_open_btn.layout.display   = "inline-flex"
            toggle_close_btn.layout.display  = "none"
            hint_bar_open.layout.display     = "block"
            hint_bar_closed.layout.display   = "none"
        else:
            body.layout.display              = "none"
            toggle_open_btn.layout.display   = "none"
            toggle_close_btn.layout.display  = "inline-flex"
            hint_bar_open.layout.display     = "none"
            hint_bar_closed.layout.display   = "block"

    toggle_open_btn.on_click(_toggle2)
    toggle_close_btn.on_click(_toggle2)

    hint_row = widgets.HBox(
        [toggle_open_btn, toggle_close_btn],
        layout=widgets.Layout(
            background_color="#0f172a",
            padding="6px 20px 10px",
            border_radius="0 0 14px 14px",
            border="1px solid rgba(255,255,255,0.07)",
            border_top="none",
            justify_content="flex-end"
        )
    )

    return widgets.VBox(
        [step_label_widget, body, hint_row],
        layout=widgets.Layout(width="100%", margin="0 0 18px 0")
    )


def _sub(title, children, tip=None, collapsible=False, open_by_default=True):
    """
    Sub-section inside a step card.
    If collapsible=True, renders as a clickable toggle panel with show/hide button.
    If collapsible=False, renders as a plain labelled section (always visible).
    """
    tip_html = f'<div style="font-size:12px;color:#363636;margin-top:4px;line-height:1.5;">💡 {tip}</div>' if tip else ""

    if not collapsible:
        plain_tip = f'<div style="font-size:12px;color:#363636;margin-bottom:8px;line-height:1.5;">💡 {tip}</div>' if tip else ""
        hdr = widgets.HTML(f"""
        <div style="font-size:13px;font-weight:700;color:#334155;
                    border-left:3px solid #3b82f6;padding-left:10px;
                    margin:14px 0 8px;">{title}</div>
        {plain_tip}""")
        return widgets.VBox([hdr] + children)

    # ── collapsible panel ──────────────────────────────────────────────────────
    state = {"open": open_by_default}

    show_btn = widgets.Button(
        description="▲  Hide",
        layout=widgets.Layout(width="80px", height="26px"),
        style={"font_size": "11px"}
    )
    hide_btn = widgets.Button(
        description="▼  Show",
        layout=widgets.Layout(width="80px", height="26px"),
        style={"font_size": "11px"}
    )
    show_btn.layout.display = "inline-flex" if open_by_default else "none"
    hide_btn.layout.display = "none" if open_by_default else "inline-flex"

    header_widget = widgets.HTML(f"""
    <div style="flex:1;">
      <div style="font-size:13px;font-weight:800;letter-spacing:0.2px;">{title}</div>
      {tip_html}
    </div>""")

    header_row = widgets.HBox(
        [header_widget, show_btn, hide_btn],
        layout=widgets.Layout(
            background_color="#1e3a5f",
            border_radius="10px 10px 0 0" if open_by_default else "10px",
            padding="10px 14px",
            align_items="center",
            margin="8px 0 0 0",
            width="100%"
        )
    )

    body_box = widgets.VBox(
        children,
        layout=widgets.Layout(
            display="flex" if open_by_default else "none",
            border="1px solid #dbe4f0",
            border_top="none",
            border_radius="0 0 10px 10px",
            padding="14px 16px",
            background_color="white",
            width="100%"
        )
    )

    def _toggle_sub(b):
        state["open"] = not state["open"]
        if state["open"]:
            body_box.layout.display    = "flex"
            show_btn.layout.display    = "inline-flex"
            hide_btn.layout.display    = "none"
            header_row.layout.border_radius = "10px 10px 0 0"
        else:
            body_box.layout.display    = "none"
            show_btn.layout.display    = "none"
            hide_btn.layout.display    = "inline-flex"
            header_row.layout.border_radius = "10px"

    show_btn.on_click(_toggle_sub)
    hide_btn.on_click(_toggle_sub)

    return widgets.VBox([header_row, body_box],
                        layout=widgets.Layout(width="100%", margin="4px 0"))


# ---- build the function selector grid ----
function_grid = widgets.VBox([
    widgets.HBox([select_all_functions_btn, clear_all_functions_btn],
                 layout=widgets.Layout(margin="0 0 10px 0")),
    widgets.VBox(_fn_rows)
], layout=widgets.Layout(
    border="1px solid #e2e8f0", border_radius="10px",
    padding="14px", background_color="white"
))

# ---- STEP 1 ----
step1 = _accordion_step("1", "Step 1 of 4", "Scenario Setup",
    "Give your scenario a name before adding groups.",
    [
        _sub("Scenario identity", [
            widgets.HBox([scenario_name_text, validate_scenario_btn]),
            global_notes_text,
        ], tip="Each scenario can contain multiple groups of parcels with different rules."),
        status_badge_out,
        toast_out,
    ], accent="#22d3ee", open_by_default=False)

# ---- STEP 2 ----
step2 = _accordion_step("2", "Step 2 of 4", "Group Editor",
    "Create or edit a group of parcels and choose which functions are allowed.",
    [
        _sub("Load or create a group", [
            widgets.HBox([saved_group_dropdown, load_group_btn, new_group_btn],
                         layout=widgets.Layout(flex_wrap="wrap", gap="6px")),
            widgets.HBox([confirm_load_btn, cancel_load_btn]),
        ], tip="A group is a set of parcels that share the same planning rules. You can have multiple groups per scenario."),
        _sub("Group actions", [
            widgets.HBox([save_group_btn, save_as_new_btn, duplicate_group_btn,
                          delete_group_btn, revert_group_btn],
                         layout=widgets.Layout(flex_wrap="wrap", gap="6px")),
            delete_confirm_box,
        ]),
        _sub("Edit history", [
            widgets.HBox([undo_btn, redo_btn],
                         layout=widgets.Layout(gap="8px")),
        ], tip="Undo / Redo lets you step back through your last 10 changes within this session."),
        _sub("Group name", [
            group_name_text,
        ]),
        _sub("Parcel IDs", [
            parcel_editor_text,
            parcel_count_out,
            widgets.HBox([parcel_sort_btn, parcel_dedupe_btn, parcel_clear_btn],
                         layout=widgets.Layout(gap="6px", margin="6px 0")),
        ], tip="Enter the IDs of the parcels you want to include. You can paste a list from Excel."),
        _sub("Parcel metrics preview", [parcel_metrics_out]),
        _sub("Allowed functions", [function_grid],
            tip="Select the land-use functions that the optimiser is allowed to assign to parcels in this group."),
        _sub("Group notes (optional)", [group_notes_text]),
    ], accent="#60a5fa", open_by_default=False)

# ---- STEP 3 ----
step3 = _accordion_step("3", "Step 3 of 4", "Planning Constraints",
    "Optionally set area limits, FAR, function rules, and advanced controls.",
    [
        _sub("📐  Development area", [
            widgets.HBox([area_mode_dropdown]),
            area_bounds_box,
            budget_box,
        ], tip="Choose how much total built area is allowed. 'Set a range' lets you define min and max. 'Let the optimiser decide' gives it full freedom (you can still cap the total with a budget).",
        collapsible=True, open_by_default=False),

        _sub("📊  Floor-Area Ratio (FAR)", [
            widgets.HBox([use_far_checkbox]),
            far_mode_box,
            group_far_box,
            parcel_far_box,
            far_rules_out,
        ], tip="FAR = total built area ÷ ground area. E.g. FAR 2.0 means you can build twice the footprint. Only available if your dataset includes ground area.",
        collapsible=True, open_by_default=False),

        _sub("🏗️  Function area limits", [
            widgets.HBox([use_function_constraints_checkbox]),
            function_mode_box,
            group_function_rule_box,
            parcel_function_rule_box,
            function_rules_out,
        ], tip="Restrict how much floor area a specific function can occupy — e.g. at least 200 m² of Residential, at most 1000 m².",
        collapsible=True, open_by_default=False),

        _sub("🔢  Active function count", [
            widgets.HBox([min_active_functions_text, max_active_functions_text],
                         layout=widgets.Layout(gap="10px")),
        ], tip="Limit how many distinct functions can be active in a parcel. E.g. max 3 means a parcel won't mix more than 3 uses.",
        collapsible=True, open_by_default=False),

        _sub("♻️  Existing use strategy", [
            widgets.VBox([preserve_existing_checkbox,
                          allow_new_functions_checkbox,
                          allow_remove_existing_checkbox]),
        ], tip="Controls whether the optimiser can add new uses, remove current ones, or must keep them.",
        collapsible=True, open_by_default=False),

        _sub("🔒  Locked parcels  (optional)", [
            locked_parcels_text,
        ], tip="Parcels in this list will be frozen exactly as they are today — the optimiser won't touch them.",
        collapsible=True, open_by_default=False),

        _sub("📍  Function presence requirements  (optional)", [
            widgets.HBox([fp_function_dropdown, fp_function_required_count_text,
                          fp_add_btn, fp_remove_btn],
                         layout=widgets.Layout(flex_wrap="wrap", gap="6px")),
            function_presence_out,
        ], tip="Require a function to appear in at least N parcels in this group. E.g. Green Space must appear in at least 2 parcels.",
        collapsible=True, open_by_default=False),
    ], accent="#a855f7", open_by_default=False)

# ---- BOTTOM SAVE BAR (between step 3 and step 4) ----
bottom_save_bar = widgets.VBox([
    widgets.HTML("""
    <div style="background:linear-gradient(135deg,#052e16 0%,#14532d 100%);
                color:white;border-radius:16px;padding:20px 26px;
                text-align:center;margin:4px 0 12px 0;">
      <div style="font-size:19px;font-weight:900;margin-bottom:6px;">
        🏙️ Done with this group?
      </div>
      <div style="font-size:13px;opacity:0.85;line-height:1.7;">
        Save it now before moving to Review &amp; Launch.<br>
        You can always come back, load it, and edit it again.
      </div>
    </div>"""),
    widgets.HBox([
        save_group_btn_bottom,
        save_as_new_btn_bottom,
        reset_editor_btn_bottom,
    ], layout=widgets.Layout(gap="12px", justify_content="center", margin="0 0 6px 0")),
    toast_out_bottom,
], layout=widgets.Layout(width="100%", margin="0 0 20px 0"))

# ---- STEP 4 ----
step4 = _accordion_step("4", "Step 4 of 4", "Review & Launch",
    "Validate your scenario, review all groups, then run Cell 4.",
    [
        launch_out,
        _sub("Diagnostics", [diagnostics_out],
             tip="Fix all errors before running Cell 4. Warnings are advisory."),
        _sub("Saved groups", [saved_groups_out]),
        _sub("Compiled parcel preview", [compiled_preview_out],
             tip="This is exactly what Cell 4 will receive — one row per parcel with all constraints applied."),
        _sub("Restore deleted groups", [trash_dropdown_out]),
    ], accent="#22c55e", open_by_default=False)

# ============================================================
# HEADER BANNER
# ============================================================
header_banner = widgets.HTML("""
<div style="background:linear-gradient(135deg,#0b1120 0%,#0f2644 40%,#312e81 100%);
            color:white;padding:24px 28px;border-radius:18px;
            box-shadow:0 12px 36px rgba(15,23,42,0.3);margin-bottom:20px;">
  <div style="font-size:26px;font-weight:900;letter-spacing:-0.5px;">
    🏙️ PCM Scenario Studio
  </div>
  <div style="font-size:13px;opacity:0.85;margin-top:6px;max-width:600px;line-height:1.6;">
    Build land-use optimisation scenarios step by step. Work through the four steps below,
    then run <b>Cell 4</b> to generate the top 10 diverse planning scenarios.
  </div>
  <div style="margin-top:14px;display:flex;gap:10px;flex-wrap:wrap;font-size:12px;opacity:0.75;">
    <span>① Scenario Setup</span><span>→</span>
    <span>② Group Editor</span><span>→</span>
    <span>③ Planning Constraints</span><span>→</span>
    <span>④ Review &amp; Launch</span>
  </div>
</div>""")

# ============================================================
# MAIN UI
# ============================================================
main_ui = widgets.VBox([
    header_banner,
    step1,
    step2,
    step3,
    bottom_save_bar,
    step4,
], layout=widgets.Layout(width="100%", max_width="920px"))

# ============================================================
# INITIAL RENDER
# ============================================================
refresh_saved_group_dropdown()
reset_editor(new_group_name=f"Group {len(scenario_config['groups']) + 1}")
update_global_settings()
compile_full_scenario()
refresh_area_mode_visibility()
refresh_far_visibility()
refresh_function_rule_visibility()
refresh_status_badge()
render_parcel_metrics()
render_far_rules_table()
render_function_rules_table()
render_function_presence_table()
render_saved_groups_panel()
render_compiled_preview()
render_diagnostics()
render_launch_panel()
render_trash_dropdown()
_refresh_undo_redo_buttons()

display(main_ui)

In [ ]:
# ============================================================
# CELL 4 — RUN OPTIMISATION
# ============================================================

import os
import time
import random
from copy import deepcopy

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

# ------------------------------------------------------------
# 0) CHECK REQUIRED OBJECTS
# ------------------------------------------------------------
required_objects = ["data_bundle", "scenario_config", "parcels_df", "RANDOM_SEED"]
for obj in required_objects:
    if obj not in globals():
        raise RuntimeError(f"Required object '{obj}' not found. Run previous cells first.")

# ------------------------------------------------------------
# 1) UNPACK DATA
# ------------------------------------------------------------
parcel_ids = data_bundle["parcel_ids"]
parcel_index = data_bundle["parcel_index"]
function_cols = data_bundle["function_cols"]
func_idx = data_bundle["func_idx"]

BASE_MATRIX = np.array(data_bundle["BASE_MATRIX"], dtype=float).copy()
parcel_total_functions_area = np.array(data_bundle["parcel_total_functions_area"], dtype=float)

parcel_ground_area = data_bundle.get("parcel_ground_area", None)
if parcel_ground_area is not None:
    parcel_ground_area = np.array(parcel_ground_area, dtype=float)

visit_freq_vec = np.array(data_bundle["visit_freq_vec"], dtype=float)
comp_weight_matrix = np.array(data_bundle["comp_weight_matrix"], dtype=float)

neighbors_idx = data_bundle["neighbors_idx"]
neighbors_dist_w = data_bundle["neighbors_dist_w"]

N = len(parcel_ids)
M = len(function_cols)

# ------------------------------------------------------------
# 2) SETTINGS + COMPILED SCENARIO
# ------------------------------------------------------------
global_settings = scenario_config.get("global_settings", {})
compiled = scenario_config.get("compiled", {})

parcel_constraints = compiled.get("parcel_constraints", {})
target_parcel_ids = compiled.get("target_parcel_ids", [])
validation_summary = compiled.get("validation_summary", {})

# Fixed run size
POP_SIZE = 200
N_GENERATIONS = 200
N_SOLUTIONS = 10

# More diversity-friendly defaults
MUTATION_RATE = float(global_settings.get("mutation_rate", 0.30))
CROSSOVER_RATE = float(global_settings.get("crossover_rate", 0.75))
ELITE_FRACTION = float(global_settings.get("elite_fraction", 0.05))
OBJECTIVE_MODE = global_settings.get("objective_mode", "maximize_complementarity")
SCENARIO_NAME = global_settings.get("scenario_name", "Scenario 1")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

if len(parcel_constraints) == 0 or len(target_parcel_ids) == 0:
    raise ValueError("No compiled parcel constraints found. Save at least one valid group in Cell 3.")

if validation_summary.get("errors"):
    raise ValueError(
        "Compiled scenario contains validation errors. Fix Cell 3 first.\n"
        + "\n".join([f"- {e}" for e in validation_summary["errors"][:30]])
    )

target_indices = []
for pid in target_parcel_ids:
    if pid not in parcel_index:
        raise ValueError(f"Parcel '{pid}' from compiled scenario does not exist in the parcels sheet.")
    c = parcel_constraints.get(pid, {})
    if not c.get("feasible", False):
        raise ValueError(f"Parcel '{pid}' is marked infeasible in compiled constraints.")
    target_indices.append(parcel_index[pid])

affected_indices = set(target_indices)
for i in target_indices:
    for nb in neighbors_idx[i]:
        affected_indices.add(int(nb))
affected_indices = sorted(list(affected_indices))

# ------------------------------------------------------------
# 3) HELPERS
# ------------------------------------------------------------
positive_total_areas = parcel_total_functions_area[parcel_total_functions_area > 0]
GLOBAL_DEFAULT_TOTAL = float(np.median(positive_total_areas)) if len(positive_total_areas) > 0 else 100.0

def clip_to_nonnegative(x):
    return np.maximum(np.array(x, dtype=float), 0.0)

def safe_float_or_none(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    return float(x)

def get_default_total_area(i):
    base_total = float(parcel_total_functions_area[i])
    if base_total > 0:
        return base_total

    row_sum = float(BASE_MATRIX[i].sum())
    if row_sum > 0:
        return row_sum

    return GLOBAL_DEFAULT_TOTAL

def parcel_constraint_for_index(i):
    pid = parcel_ids[i]
    if pid not in parcel_constraints:
        raise KeyError(f"Parcel '{pid}' not found in compiled parcel constraints.")
    return parcel_constraints[pid]

def allowed_function_indices_for_parcel_constraint(pc):
    allowed_functions = pc.get("allowed_functions", []) or []
    return [func_idx[f] for f in allowed_functions if f in func_idx]

def function_bounds_arrays(pc):
    mins = np.zeros(M, dtype=float)
    maxs = np.zeros(M, dtype=float)

    bounds = pc.get("function_bounds", {})
    for j, fname in enumerate(function_cols):
        rule = bounds.get(fname, {"min_area": 0.0, "max_area": 0.0})
        rmin = safe_float_or_none(rule.get("min_area"))
        rmax = safe_float_or_none(rule.get("max_area"))

        mins[j] = 0.0 if rmin is None else max(0.0, float(rmin))
        if rmax is None:
            maxs[j] = np.inf
        else:
            maxs[j] = max(0.0, float(rmax))

    maxs = np.maximum(maxs, mins)
    return mins, maxs

def total_bounds_for_parcel(i, pc):
    lo = safe_float_or_none(pc.get("total_area_min"))
    hi = safe_float_or_none(pc.get("total_area_max"))

    if lo is None:
        lo = 0.0

    if hi is None:
        hi = max(lo, get_default_total_area(i))

    if hi < lo:
        hi = lo

    return float(lo), float(hi)

def choose_target_total(i, pc):
    lo, hi = total_bounds_for_parcel(i, pc)
    if abs(hi - lo) < 1e-9:
        return float(lo)
    return float(random.uniform(lo, hi))

def finite_capacity_sum(maxs):
    finite_mask = np.isfinite(maxs)
    return float(np.sum(maxs[finite_mask]))

def round_row_to_integers_with_total(row, target_total=None):
    row = np.array(row, dtype=float)
    row = np.maximum(row, 0.0)

    floored = np.floor(row)
    remainder = row - floored

    if target_total is None:
        target_total = int(round(row.sum()))
    else:
        target_total = int(round(target_total))

    current_total = int(floored.sum())
    deficit = target_total - current_total

    if deficit > 0:
        order = np.argsort(-remainder)
        assigned = 0
        for idx in order:
            floored[idx] += 1
            assigned += 1
            if assigned >= deficit:
                break
    elif deficit < 0:
        order = np.argsort(remainder)
        removed = 0
        for idx in order:
            if floored[idx] > 0:
                floored[idx] -= 1
                removed += 1
                if removed >= abs(deficit):
                    break

    return floored.astype(float)

def allocate_remaining_with_caps(base_vec, remaining, capacity_vec, priority_weights=None):
    out = np.array(base_vec, dtype=float).copy()
    caps = np.maximum(np.array(capacity_vec, dtype=float), 0.0)

    if remaining <= 1e-12 or np.sum(caps) <= 1e-12:
        return out

    active = np.where(caps > 1e-12)[0]
    if len(active) == 0:
        return out

    if priority_weights is None:
        weights = np.ones(len(active), dtype=float)
    else:
        w = np.array(priority_weights, dtype=float)
        weights = np.maximum(w[active], 0.0)
        if weights.sum() <= 1e-12:
            weights = np.ones(len(active), dtype=float)

    weights = weights / weights.sum()
    rem = float(remaining)

    active_list = list(active)
    current_weights = weights.copy()

    while rem > 1e-9 and len(active_list) > 0:
        idx = np.array(active_list, dtype=int)
        raw = rem * (current_weights[:len(idx)] / current_weights[:len(idx)].sum())
        give = np.minimum(raw, caps[idx])
        out[idx] += give
        caps[idx] -= give
        rem -= float(np.sum(give))

        active_list = [k for k in active_list if caps[k] > 1e-9]
        if len(active_list) > 0:
            current_weights = np.ones(len(active_list), dtype=float)

    return out

def fit_row_to_bounds_and_total(row, mins, maxs, target_total):
    row = clip_to_nonnegative(row)
    mins = np.array(mins, dtype=float)
    maxs = np.array(maxs, dtype=float)

    row = np.maximum(row, mins)
    finite_mask = np.isfinite(maxs)
    row[finite_mask] = np.minimum(row[finite_mask], maxs[finite_mask])

    min_sum = float(np.sum(mins))
    max_sum = finite_capacity_sum(maxs)
    if np.any(~np.isfinite(maxs)):
        max_sum = np.inf

    if target_total <= min_sum + 1e-9:
        out = mins.copy()
        out = round_row_to_integers_with_total(out, target_total=min_sum)
        out = np.maximum(out, mins)
        out[finite_mask] = np.minimum(out[finite_mask], maxs[finite_mask])
        return clip_to_nonnegative(out)

    if np.isfinite(max_sum) and target_total >= max_sum - 1e-9:
        out = mins.copy()
        caps = np.maximum(maxs - mins, 0.0)
        out = allocate_remaining_with_caps(out, max_sum - min_sum, caps)
        out = round_row_to_integers_with_total(out, target_total=max_sum)
        out = np.maximum(out, mins)
        out[finite_mask] = np.minimum(out[finite_mask], maxs[finite_mask])
        return clip_to_nonnegative(out)

    out = mins.copy()
    remaining = float(target_total - min_sum)

    extra_pref = np.maximum(row - mins, 0.0)
    if np.sum(extra_pref) <= 1e-12:
        extra_pref = np.random.random(len(row))

    caps = maxs - mins
    caps[~np.isfinite(caps)] = remaining
    caps = np.maximum(caps, 0.0)

    out = allocate_remaining_with_caps(out, remaining, caps, priority_weights=extra_pref)
    out = clip_to_nonnegative(out)
    out = round_row_to_integers_with_total(out, target_total=target_total)

    out = np.maximum(out, mins)
    out[finite_mask] = np.minimum(out[finite_mask], maxs[finite_mask])

    target_total_int = int(round(np.clip(target_total, np.sum(mins), 1e18)))
    out = round_row_to_integers_with_total(out, target_total=target_total_int)
    out = np.maximum(out, mins)
    out[finite_mask] = np.minimum(out[finite_mask], maxs[finite_mask])

    return clip_to_nonnegative(out)

def random_row_for_parcel(i, pc):
    mins, maxs = function_bounds_arrays(pc)
    allowed_idx = allowed_function_indices_for_parcel_constraint(pc)

    if len(allowed_idx) == 0:
        return mins.copy()

    target_total = choose_target_total(i, pc)

    seed = mins.copy()
    extra_caps = np.maximum(maxs - mins, 0.0)
    extra_caps[~np.isfinite(extra_caps)] = target_total

    positive_cap_idx = [j for j in allowed_idx if extra_caps[j] > 1e-9]
    if len(positive_cap_idx) > 0:
        max_pick = max(1, min(len(positive_cap_idx), max(2, len(positive_cap_idx) // 2)))
        k = random.randint(1, max_pick)
        chosen = random.sample(positive_cap_idx, k=k)
        priority = np.zeros(M, dtype=float)
        priority[chosen] = np.random.random(len(chosen)) + 0.1
    else:
        priority = np.zeros(M, dtype=float)

    row = fit_row_to_bounds_and_total(seed + priority, mins, maxs, target_total)
    return row

def repair_row(i, row, pc):
    mins, maxs = function_bounds_arrays(pc)
    lo, hi = total_bounds_for_parcel(i, pc)
    clipped_total = float(np.clip(np.sum(row), lo, hi))
    row = fit_row_to_bounds_and_total(row, mins, maxs, clipped_total)

    current_total = float(np.sum(row))
    if current_total < lo - 1e-8 or current_total > hi + 1e-8:
        row = fit_row_to_bounds_and_total(row, mins, maxs, choose_target_total(i, pc))

    row = clip_to_nonnegative(row)
    row = round_row_to_integers_with_total(row, target_total=row.sum())
    return clip_to_nonnegative(row)

def score_parcel_interparcel(i, matrix):
    row_i = matrix[i]
    total_i = row_i.sum()
    if total_i <= 0:
        return 0.0

    nz_i = np.where(row_i > 0)[0]
    if len(nz_i) == 0:
        return 0.0

    score = 0.0
    for k, nb in enumerate(neighbors_idx[i]):
        row_nb = matrix[nb]
        nz_nb = np.where(row_nb > 0)[0]
        if len(nz_nb) == 0:
            continue

        dist_w = neighbors_dist_w[i][k]

        for a in nz_i:
            ai = row_i[a] / total_i
            vf = visit_freq_vec[a]
            for b in nz_nb:
                score += ai * float(row_nb[b]) * comp_weight_matrix[a, b] * vf * dist_w

    return float(score)

def fitness(matrix):
    return float(sum(score_parcel_interparcel(i, matrix) for i in affected_indices))

def build_candidate_from_genes(genes_dict):
    matrix = BASE_MATRIX.copy()
    for i, row in genes_dict.items():
        matrix[i] = row
    return matrix

def random_individual():
    genes = {}
    for i in target_indices:
        pc = parcel_constraint_for_index(i)
        genes[i] = random_row_for_parcel(i, pc)
    return genes

def crossover(parent_a, parent_b):
    child = {}
    for i in target_indices:
        if random.random() < 0.5:
            child[i] = parent_a[i].copy()
        else:
            child[i] = parent_b[i].copy()
    return child

def mutate(genes):
    child = {i: genes[i].copy() for i in genes}

    for i in target_indices:
        if random.random() < MUTATION_RATE:
            pc = parcel_constraint_for_index(i)
            row = child[i].copy()

            if random.random() < 0.20:
                row = random_row_for_parcel(i, pc)
            else:
                mins, maxs = function_bounds_arrays(pc)
                allowed_idx = allowed_function_indices_for_parcel_constraint(pc)

                noise = np.random.normal(loc=0.0, scale=0.18, size=M)
                row = row * (1.0 + noise)
                row = clip_to_nonnegative(row)

                if len(allowed_idx) > 0 and random.random() < 0.45:
                    j = random.choice(allowed_idx)
                    if row[j] > 1e-9:
                        row[j] *= random.uniform(0.0, 0.5)
                    else:
                        inject = max(1.0, 0.05 * max(1.0, row.sum()))
                        row[j] = inject

                if len(allowed_idx) > 1 and random.random() < 0.30:
                    j_from = random.choice(allowed_idx)
                    j_to = random.choice([x for x in allowed_idx if x != j_from])
                    move_amt = min(row[j_from], random.uniform(10.0, max(10.0, 0.15 * max(1.0, row.sum()))))
                    row[j_from] = max(0.0, row[j_from] - move_amt)
                    row[j_to] += move_amt

                lo, hi = total_bounds_for_parcel(i, pc)
                current_total = float(np.sum(row))
                target_total = np.clip(current_total * random.uniform(0.88, 1.12), lo, hi)
                row = fit_row_to_bounds_and_total(row, mins, maxs, target_total)
                row = repair_row(i, row, pc)

            child[i] = row

    return child

def tournament_select(scored_population, k=3):
    picks = random.sample(scored_population, k=min(k, len(scored_population)))
    picks = sorted(picks, key=lambda x: x["fitness"], reverse=True)
    return picks[0]["genes"]

def genes_signature_from_matrix(matrix):
    sig_parts = []
    for i in target_indices:
        sig_parts.append(tuple(np.round(matrix[i], 0).astype(int)))
    return tuple(sig_parts)

def solution_distance_matrix(mat_a, mat_b):
    dist = 0.0
    for i in target_indices:
        dist += np.sum(np.abs(mat_a[i] - mat_b[i]))
    return float(dist)

def is_meaningfully_different(candidate_matrix, archive_list, min_abs_diff=500.0):
    if len(archive_list) == 0:
        return True
    for item in archive_list:
        d = solution_distance_matrix(candidate_matrix, item["matrix"])
        if d < min_abs_diff:
            return False
    return True

# ------------------------------------------------------------
# 4) FUTURISTIC PROGRESS UI
# ------------------------------------------------------------
progress_out = widgets.Output()
display(progress_out)

def render_progress_card(gen, total_gen, best_fit, mean_fit, elapsed_now, stage_text, completed=False, archive_size=0):
    progress_pct = 100.0 * gen / total_gen if total_gen > 0 else 0.0
    card_bg = (
        "linear-gradient(135deg,#052e16 0%,#14532d 40%,#064e3b 100%)"
        if completed else
        "linear-gradient(135deg,#0b1020 0%,#16213e 45%,#312e81 100%)"
    )

    title = "Optimisation complete" if completed else "PCM Optimisation Engine"

    with progress_out:
        clear_output(wait=True)
        display(HTML(f"""
        <div style="
            background:{card_bg};
            color:white;
            padding:22px 26px;
            border-radius:24px;
            box-shadow:0 18px 40px rgba(15,23,42,0.22);
            margin:12px 0;
            border:1px solid rgba(255,255,255,0.08);">
            <div style="font-size:24px; font-weight:800; margin-bottom:10px;">
                {title}
            </div>
            <div style="font-size:13px; opacity:0.92; margin-bottom:16px; line-height:1.6;">
                {stage_text}
            </div>

            <div style="display:flex; gap:10px; flex-wrap:wrap; margin-bottom:14px;">
                <span style="padding:6px 10px; border-radius:999px; background:#1d4ed8;">Scenario: {SCENARIO_NAME}</span>
                <span style="padding:6px 10px; border-radius:999px; background:#0f766e;">Generation: {gen} / {total_gen}</span>
                <span style="padding:6px 10px; border-radius:999px; background:#7c3aed;">Population: {POP_SIZE}</span>
                <span style="padding:6px 10px; border-radius:999px; background:#166534;">Best: {best_fit:,.2f}</span>
                <span style="padding:6px 10px; border-radius:999px; background:#92400e;">Mean: {mean_fit:,.2f}</span>
                <span style="padding:6px 10px; border-radius:999px; background:#334155;">Elapsed: {elapsed_now:.1f} sec</span>
                <span style="padding:6px 10px; border-radius:999px; background:#be185d;">Archive: {archive_size}</span>
            </div>

            <div style="background:rgba(255,255,255,0.10); border-radius:999px; height:16px; overflow:hidden; margin-bottom:10px;">
                <div style="
                    width:{progress_pct:.1f}%;
                    height:100%;
                    background:linear-gradient(90deg,#22d3ee 0%, #60a5fa 45%, #a78bfa 100%);
                    box-shadow:0 0 14px rgba(96,165,250,0.7);">
                </div>
            </div>

            <div style="font-size:12px; opacity:0.88;">
                Progress: {progress_pct:.1f}%
            </div>
        </div>
        """))

# ------------------------------------------------------------
# 5) BASELINE
# ------------------------------------------------------------
baseline_matrix = BASE_MATRIX.copy()
baseline_fitness = fitness(baseline_matrix)

render_progress_card(
    gen=0,
    total_gen=N_GENERATIONS,
    best_fit=baseline_fitness,
    mean_fit=baseline_fitness,
    elapsed_now=0.0,
    stage_text="Preparing the optimisation run, checking compiled parcel constraints, and building the initial population.",
    archive_size=0
)

# ------------------------------------------------------------
# 6) INITIAL POPULATION
# ------------------------------------------------------------
t0 = time.time()
population = []

# ------------------------------------------------------------
# 6B) GLOBAL ARCHIVE OF DIVERSE GOOD SOLUTIONS
# ------------------------------------------------------------
ARCHIVE_MAX_SIZE = 80
MIN_SCENARIO_DISTANCE = 500.0

solution_archive = []

def try_add_to_archive(genes, fit):
    global solution_archive

    matrix = build_candidate_from_genes(genes)
    signature = genes_signature_from_matrix(matrix)

    for item in solution_archive:
        if item["signature"] == signature:
            return

    if not is_meaningfully_different(matrix, solution_archive, min_abs_diff=MIN_SCENARIO_DISTANCE):
        return

    solution_archive.append({
        "genes": {i: genes[i].copy() for i in genes},
        "matrix": matrix.copy(),
        "fitness": float(fit),
        "signature": signature
    })

    solution_archive = sorted(solution_archive, key=lambda x: x["fitness"], reverse=True)[:ARCHIVE_MAX_SIZE]

for n in range(POP_SIZE):
    genes = random_individual()
    matrix = build_candidate_from_genes(genes)
    fit = fitness(matrix)
    population.append({"genes": genes, "fitness": fit})
    try_add_to_archive(genes, fit)

    if (n + 1) == 1 or (n + 1) % 20 == 0 or (n + 1) == POP_SIZE:
        best_init = max(x["fitness"] for x in population)
        mean_init = float(np.mean([x["fitness"] for x in population]))
        render_progress_card(
            gen=0,
            total_gen=N_GENERATIONS,
            best_fit=best_init,
            mean_fit=mean_init,
            elapsed_now=time.time() - t0,
            stage_text=f"Building initial population: {n + 1} / {POP_SIZE} individuals evaluated.",
            archive_size=len(solution_archive)
        )

history = []

# ------------------------------------------------------------
# 7) EVOLUTION LOOP
# ------------------------------------------------------------
elite_n = max(1, int(round(ELITE_FRACTION * POP_SIZE)))

for gen in range(1, N_GENERATIONS + 1):
    population = sorted(population, key=lambda x: x["fitness"], reverse=True)

    for item in population[:20]:
        try_add_to_archive(item["genes"], item["fitness"])

    best_fit = population[0]["fitness"]
    mean_fit = float(np.mean([x["fitness"] for x in population]))
    history.append({
        "generation": gen,
        "best_fitness": best_fit,
        "mean_fitness": mean_fit
    })

    render_progress_card(
        gen=gen,
        total_gen=N_GENERATIONS,
        best_fit=best_fit,
        mean_fit=mean_fit,
        elapsed_now=time.time() - t0,
        stage_text="Running evolutionary search, repairing candidate rows lightly, and preserving diverse high-performing solutions.",
        archive_size=len(solution_archive)
    )

    new_population = population[:elite_n]

    RANDOM_IMMIGRANTS = max(6, int(0.08 * POP_SIZE))
    target_offspring_n = POP_SIZE - elite_n - RANDOM_IMMIGRANTS

    while len(new_population) < elite_n + target_offspring_n:
        p1 = tournament_select(population)
        p2 = tournament_select(population)

        if random.random() < CROSSOVER_RATE:
            child_genes = crossover(p1, p2)
        else:
            child_genes = {i: p1[i].copy() for i in p1}

        child_genes = mutate(child_genes)
        child_matrix = build_candidate_from_genes(child_genes)
        child_fit = fitness(child_matrix)

        new_population.append({"genes": child_genes, "fitness": child_fit})
        try_add_to_archive(child_genes, child_fit)

    for _ in range(RANDOM_IMMIGRANTS):
        genes = random_individual()
        matrix = build_candidate_from_genes(genes)
        fit = fitness(matrix)
        new_population.append({"genes": genes, "fitness": fit})
        try_add_to_archive(genes, fit)

    population = new_population[:POP_SIZE]

elapsed = time.time() - t0

# ------------------------------------------------------------
# 8) COLLECT TOP UNIQUE SOLUTIONS FROM GLOBAL ARCHIVE
# ------------------------------------------------------------
if len(solution_archive) == 0:
    raise RuntimeError("No archived solutions were generated.")

top_unique = solution_archive[:N_SOLUTIONS]

best_solution = top_unique[0]
best_matrix = best_solution["matrix"]
best_fitness = best_solution["fitness"]

# ------------------------------------------------------------
# 9) BUILD SCENARIO SHEETS
# ------------------------------------------------------------
scenario_sheet_data = {}

for rank, sol in enumerate(top_unique, start=1):
    mat = sol["matrix"]
    fit = sol["fitness"]

    allocation_rows = []
    for i in target_indices:
        pid = parcel_ids[i]
        pc = parcel_constraint_for_index(i)

        base_total = float(BASE_MATRIX[i].sum())
        new_total = float(mat[i].sum())

        if parcel_ground_area is not None:
            garea = float(parcel_ground_area[i]) if np.isfinite(parcel_ground_area[i]) else np.nan
        else:
            garea = np.nan

        if pd.notna(garea) and garea > 0:
            base_far = base_total / garea
            new_far = new_total / garea
            far_change = new_far - base_far
        else:
            base_far = np.nan
            new_far = np.nan
            far_change = np.nan

        baseline_local_score = score_parcel_interparcel(i, BASE_MATRIX)
        new_local_score = score_parcel_interparcel(i, mat)

        row = {
            "scenario_rank": rank,
            "scenario_fitness": round(float(fit), 4),
            "fitness_gain_vs_baseline": round(float(fit - baseline_fitness), 4),
            "group_name": pc.get("group_name"),
            "parcel_id": pid,
            "ground_area": np.nan if pd.isna(garea) else int(round(garea)),
            "base_total_area": int(round(base_total)),
            "new_total_area": int(round(new_total)),
            "delta_total_area": int(round(new_total - base_total)),
            "base_active_functions": int(np.sum(BASE_MATRIX[i] > 0)),
            "new_active_functions": int(np.sum(mat[i] > 0)),
            "base_far": np.nan if pd.isna(base_far) else round(float(base_far), 4),
            "new_far": np.nan if pd.isna(new_far) else round(float(new_far), 4),
            "far_change": np.nan if pd.isna(far_change) else round(float(far_change), 4),
            "baseline_local_score": round(float(baseline_local_score), 4),
            "new_local_score": round(float(new_local_score), 4),
            "local_score_change": round(float(new_local_score - baseline_local_score), 4),
        }

        for j, fname in enumerate(function_cols):
            row[fname] = int(round(mat[i, j]))

        allocation_rows.append(row)

    scenario_sheet_data[f"scenario_{rank}"] = pd.DataFrame(allocation_rows)

# ------------------------------------------------------------
# 10) EXPORT PATHS
# ------------------------------------------------------------
safe_name = "".join(c if c.isalnum() or c in ("_", "-") else "_" for c in SCENARIO_NAME.strip().replace(" ", "_"))
if not safe_name:
    safe_name = "scenario"

preferred_dirs = [
    "/mnt/data",
    "/content",
    os.getcwd()
]

output_dir = None
for d in preferred_dirs:
    if os.path.isdir(d):
        output_dir = d
        break

if output_dir is None:
    output_dir = os.getcwd()

excel_output_path = os.path.join(output_dir, f"{safe_name}_PCM_top10_scenarios.xlsx")

# ------------------------------------------------------------
# 11) EXPORT EXCEL WORKBOOK
#     - only scenario sheets
#     - changed new allocations highlighted in green
# ------------------------------------------------------------
green_fill = PatternFill(fill_type="solid", start_color="C6EFCE", end_color="C6EFCE")
header_fill = PatternFill(fill_type="solid", start_color="DCE6F1", end_color="DCE6F1")
bold_font = Font(bold=True)

with pd.ExcelWriter(excel_output_path, engine="openpyxl") as writer:
    for sheet_name, df_alloc in scenario_sheet_data.items():
        safe_sheet_name = sheet_name[:31]
        df_alloc.to_excel(writer, sheet_name=safe_sheet_name, index=False)

        ws = writer.sheets[safe_sheet_name]

        for cell in ws[1]:
            cell.font = bold_font
            cell.fill = header_fill
            cell.alignment = Alignment(horizontal="center", vertical="center")

        ws.freeze_panes = "A2"

        for col_idx, col_name in enumerate(df_alloc.columns, start=1):
            if col_name in function_cols:
                width = 14
            else:
                width = min(max(len(str(col_name)) + 2, 12), 24)
            ws.column_dimensions[get_column_letter(col_idx)].width = width

        col_pos = {col_name: idx + 1 for idx, col_name in enumerate(df_alloc.columns)}

        for df_row_idx, (_, row) in enumerate(df_alloc.iterrows(), start=2):
            pid = row["parcel_id"]
            i = parcel_index[pid]

            for fname in function_cols:
                j = func_idx[fname]
                base_val = int(round(BASE_MATRIX[i, j]))
                new_val = int(row[fname])

                if new_val != base_val:
                    ws.cell(row=df_row_idx, column=col_pos[fname]).fill = green_fill

            if int(row["new_total_area"]) != int(row["base_total_area"]):
                ws.cell(row=df_row_idx, column=col_pos["new_total_area"]).fill = green_fill

            if pd.notna(row["new_far"]) and pd.notna(row["base_far"]):
                if abs(float(row["new_far"]) - float(row["base_far"])) > 1e-9:
                    ws.cell(row=df_row_idx, column=col_pos["new_far"]).fill = green_fill

            if abs(float(row["new_local_score"]) - float(row["baseline_local_score"])) > 1e-9:
                ws.cell(row=df_row_idx, column=col_pos["new_local_score"]).fill = green_fill

# ------------------------------------------------------------
# 12) FINAL PROGRESS CARD
# ------------------------------------------------------------
final_mean_fit = float(np.mean([x["fitness"] for x in population]))

render_progress_card(
    gen=N_GENERATIONS,
    total_gen=N_GENERATIONS,
    best_fit=best_fitness,
    mean_fit=final_mean_fit,
    elapsed_now=elapsed,
    stage_text=(
        f"Optimisation finished. {len(top_unique)} diverse scenario solutions were exported to Excel. "
        f"Run the next save cell to choose the file name and destination on your computer."
    ),
    completed=True,
    archive_size=len(solution_archive)
)

# ------------------------------------------------------------
# 13) COMPLETION SUMMARY
# ------------------------------------------------------------
display(HTML(f"""
<div style="
    background:linear-gradient(135deg,#052e16 0%,#14532d 50%,#064e3b 100%);
    color:white;
    padding:22px 26px;
    border-radius:18px;
    box-shadow:0 12px 32px rgba(5,46,22,0.35);
    margin:14px 0;
    border:1px solid rgba(255,255,255,0.10);">
  <div style="font-size:22px;font-weight:900;margin-bottom:10px;">
    ✅ Optimisation complete
  </div>
  <div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:16px;font-size:13px;">
    <span style="padding:6px 12px;border-radius:999px;background:#166534;">
      📊 {len(top_unique)} scenarios exported
    </span>
    <span style="padding:6px 12px;border-radius:999px;background:#1d4ed8;">
      🏆 Best fitness: {best_fitness:.4f}
    </span>
    <span style="padding:6px 12px;border-radius:999px;background:#7c3aed;">
      🗄 Archive: {len(solution_archive)} diverse solutions
    </span>
    <span style="padding:6px 12px;border-radius:999px;background:#0f766e;">
      📁 {excel_output_path}
    </span>
  </div>
  <div style="
    background:rgba(255,255,255,0.10);
    border-radius:12px;
    padding:12px 16px;
    font-size:13px;
    line-height:1.7;">
    ➡️ <b>Next step:</b> Run <b>Cell 5</b> to save the Excel file to your computer.
  </div>
</div>
"""))

In [ ]:
# ============================================================
# CELL 5 — SAVE EXCEL TO PC WITH BROWSER FILE PICKER
# ============================================================

import os
import base64
import ipywidgets as widgets
from IPython.display import display, HTML

# ------------------------------------------------------------
# 0) CHECK THAT CELL 4 ALREADY CREATED THE EXCEL FILE
# ------------------------------------------------------------
if "excel_output_path" not in globals():
    raise RuntimeError("excel_output_path was not found. Run Cell 4 first.")

if not os.path.exists(excel_output_path):
    raise RuntimeError(f"Excel file not found at: {excel_output_path}")

# ------------------------------------------------------------
# 1) READ FILE AS BASE64
# ------------------------------------------------------------
with open(excel_output_path, "rb") as f:
    file_bytes = f.read()

file_b64 = base64.b64encode(file_bytes).decode("utf-8")
default_filename = os.path.basename(excel_output_path)

# ------------------------------------------------------------
# 2) UI
# ------------------------------------------------------------
save_btn = widgets.Button(
    description="Save Excel to PC",
    icon="download",
    button_style="success",
    layout=widgets.Layout(width="220px", height="42px")
)

msg_out = widgets.Output()

display(HTML("""
<div style="
    background: linear-gradient(135deg,#0b1020 0%,#16213e 45%,#312e81 100%);
    color: white;
    padding: 18px 22px;
    border-radius: 20px;
    margin: 10px 0 14px 0;
    box-shadow: 0 12px 28px rgba(15,23,42,0.20);
    border: 1px solid rgba(255,255,255,0.08);
">
    <div style="font-size:20px; font-weight:800; margin-bottom:8px;">
        Excel export ready
    </div>
    <div style="font-size:13px; opacity:0.92; line-height:1.6;">
        Click the button below to choose the file name and the folder on your computer.
    </div>
</div>
"""))

display(save_btn, msg_out)

# ------------------------------------------------------------
# 3) CLICK HANDLER
# ------------------------------------------------------------
def _save_to_pc(_):
    with msg_out:
        msg_out.clear_output()
        display(HTML("""
        <div style="
            padding:12px 16px;
            border-radius:12px;
            background:#f0fdf4;
            color:#166534;
            border:1px solid #86efac;
            font-size:13px;
            line-height:1.7;
            margin-top:10px;">
            <b>&#9203; A save dialog should appear now.</b><br>
            Choose your folder, give the file a name, and click <b>Save</b>.<br>
            <span style="color:#475569;">
              If no dialog appeared, your browser downloaded the file automatically
              to your <b>Downloads</b> folder.
            </span>
        </div>
        """))

    js = f"""
    <script>
    (async () => {{
        const b64 = "{file_b64}";
        const defaultName = "{default_filename}";
        const mimeType = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet";

        function b64ToUint8Array(base64) {{
            const binary = atob(base64);
            const len = binary.length;
            const bytes = new Uint8Array(len);
            for (let i = 0; i < len; i++) {{
                bytes[i] = binary.charCodeAt(i);
            }}
            return bytes;
        }}

        const bytes = b64ToUint8Array(b64);
        const blob = new Blob([bytes], {{ type: mimeType }});

        try {{
            if ('showSaveFilePicker' in window) {{
                const handle = await window.showSaveFilePicker({{
                    suggestedName: defaultName,
                    types: [{{
                        description: 'Excel Workbook',
                        accept: {{
                            'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet': ['.xlsx']
                        }}
                    }}]
                }});

                const writable = await handle.createWritable();
                await writable.write(blob);
                await writable.close();
            }} else {{
                const url = URL.createObjectURL(blob);
                const a = document.createElement('a');
                a.href = url;
                a.download = defaultName;
                document.body.appendChild(a);
                a.click();
                document.body.removeChild(a);
                setTimeout(() => URL.revokeObjectURL(url), 2000);
                alert("Your browser does not support the save picker here. A normal download was triggered instead.");
            }}
        }} catch (err) {{
            console.log(err);
        }}
    }})();
    </script>
    """
    display(HTML(js))

save_btn.on_click(_save_to_pc)